# Factor Momentum on Equity Risk-Factor ETFs

An ETF-based **cross-sectional Factor Momentum** strategy inspired by **Arnott et al.**, evaluated under realistic frictions.

## What the notebook covers
- **6-1 / 12-1** cross-sectional factor-momentum signals
- **Genuine next-day implementation** (`IMPLEMENTATION_LAG = 1`) with intra-month weight drift
- **Historical daily risk-free** adjustment from **FRED** (default `DGS3MO`)
- **Proxy transaction costs** (**Abdi-Ranaldo 2017**, **Corwin-Schultz 2012**) plus a **flat-bps** sanity check
- A **passive market benchmark** (`SPY`) alongside the equal-weight factor blend
- **Politis-Romano stationary bootstrap** run on the common sample, with the strategy rebuilt inside every replica
- **Paired bootstrap differences** (Mom minus benchmark) with `P(Mom > benchmark)`
- A **random-selection (signal-permutation) null** that isolates the contribution of the signal
- **Spanning regression** (Newey-West alpha vs the factor ETFs + SPY) and **inverse-volatility** weighting that separates selection from concentration
- **Sharpe-ratio inference** (Ledoit-Wolf delta-Sharpe; Probabilistic / Deflated Sharpe), **break-even cost**, **BCa** intervals, **regime/subsample** stability
- **Serial-dependence diagnostics** (Ljung-Box / ARCH-LM) and **dependence-aware bootstrap engines** (FHS AR-GJR-GARCH and a VAR-sieve wild bootstrap)
- **Multiple-testing** control across the specification grid (White Reality Check / Hansen SPA)
- **Localising the gap**: the same signal on Fama-French **long-short** factors, a **true out-of-sample** walk-forward, **rank/quantile** weighting, **conditional (regime) spanning**, and a **static-replication + selection-quality** decomposition of the result

## Practical note
The default configuration runs everything end to end and is computationally heavy (the bootstraps, engines and SPA each resample the full pipeline). For a quick smoke test, set the `RUN_*` flags you don't need to `False` and/or lower `N_BOOT`, `N_RANDOM`, `ENGINE_N_SIMS`, `MT_N_BOOT`. The FHS engine requires the `arch` package.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None


# ============================================================
# DATA
# ============================================================

TICKERS = {
    "Value": "RPV",
    "Size": "SIZE",
    "Momentum": "MTUM",
    "Quality": "SPHQ",
    "LowVol": "SPLV",
}
FACTOR_NAMES = list(TICKERS)


def load_factor_etf_returns_hdf(filepath: str, key: str = "/FactorETFs") -> pd.DataFrame:
    """Load daily factor-ETF returns from an HDF store (5 columns)."""
    r = pd.read_hdf(filepath, key)
    r.index = pd.to_datetime(r.index)
    r.index.name = "Date"
    start = r.dropna().index[0]
    r = r.loc[start:].copy()
    if r.shape[1] == 5:
        r.columns = FACTOR_NAMES
    return r.astype(float)


def download_factor_etf_bundle(start: str = "2000-01-01") -> dict:
    """Download adjusted OHLC for the factor ETF panel via Yahoo Finance."""
    if yf is None:
        raise ImportError("yfinance is not available in this environment.")
    raw = yf.download(list(TICKERS.values()), start=start, auto_adjust=True, progress=False)
    if not isinstance(raw.columns, pd.MultiIndex):
        raise ValueError("Expected a MultiIndex Yahoo Finance download for multiple tickers.")
    close = raw["Close"].rename(columns={v: k for k, v in TICKERS.items()})
    high = raw["High"].rename(columns={v: k for k, v in TICKERS.items()})
    low = raw["Low"].rename(columns={v: k for k, v in TICKERS.items()})
    for obj in (close, high, low):
        obj.index = pd.to_datetime(obj.index)
        obj.index.name = "Date"
        obj.sort_index(inplace=True)
    rets = close.pct_change().dropna(how="all")
    rets.index.name = "Date"
    return {"close": close.astype(float), "high": high.astype(float),
            "low": low.astype(float), "rets": rets.astype(float)}


def download_benchmark_returns(ticker: str = "SPY", start: str = "2000-01-01") -> pd.Series:
    """Daily close-to-close returns of a passive market benchmark (default SPY)."""
    if yf is None:
        raise ImportError("yfinance is not available in this environment.")
    raw = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    close = raw["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    close.index = pd.to_datetime(close.index)
    rets = close.sort_index().pct_change().dropna()
    rets.name = ticker
    return rets.astype(float)


# ============================================================
# HISTORICAL RISK-FREE RATE
# ============================================================

def fetch_fred_series(series_id, start, end) -> pd.Series:
    """Download a daily FRED series and forward-fill missing business-day obs."""
    if pdr is None:
        raise ImportError("pandas_datareader is required for FRED, or set USE_HISTORICAL_RF = False.")
    s = pdr.DataReader(series_id, "fred", start, end)
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    s.index = pd.to_datetime(s.index)
    s = s.astype(float).sort_index().ffill()
    s.name = series_id
    return s


def build_historical_rf_daily(index, rf_source="DGS3MO", day_count=360, lag_days=1) -> pd.Series:
    """Historical daily simple risk-free return series aligned to `index`."""
    idx = pd.DatetimeIndex(index).sort_values().unique()
    if len(idx) == 0:
        return pd.Series(dtype=float, name="rf_daily")
    rf_source = rf_source.upper()
    if rf_source == "SOFR" and idx.min() < pd.Timestamp("2018-04-03"):
        raise ValueError("SOFR starts 2018-04-03. For a full 2013+ sample use RF_SOURCE='DGS3MO'.")
    start = idx.min() - pd.Timedelta(days=10)
    end = idx.max() + pd.Timedelta(days=10)
    y = fetch_fred_series(rf_source, start, end)
    rf_daily = (y / 100.0) / day_count
    rf_daily = rf_daily.shift(lag_days).ffill()
    rf = rf_daily.reindex(idx).ffill().bfill()
    rf.name = "rf_daily"
    return rf.astype(float)


def download_irx_rf_daily(index, day_count=360, lag_days=1) -> pd.Series:
    """Daily simple risk-free from Yahoo `^IRX` (13-week T-bill discount yield, in %).
    Fallback for when FRED is unreachable."""
    if yf is None:
        raise ImportError("yfinance is not available in this environment.")
    idx = pd.DatetimeIndex(index).sort_values().unique()
    start = idx.min() - pd.Timedelta(days=10)
    raw = yf.download("^IRX", start=str(start.date()), auto_adjust=False, progress=False)
    y = raw["Close"]
    if isinstance(y, pd.DataFrame):
        y = y.iloc[:, 0]
    y.index = pd.to_datetime(y.index)
    y = y.astype(float).sort_index().ffill()
    rf_daily = (y / 100.0) / day_count
    rf_daily = rf_daily.shift(lag_days).ffill()
    rf = rf_daily.reindex(idx).ffill().bfill()
    rf.name = "rf_daily"
    return rf.astype(float)


def build_rf_with_fallback(index, rf_source="DGS3MO", day_count=360, lag_days=1, fred_attempts=3):
    """Risk-free cascade: FRED (`rf_source`) -> Yahoo `^IRX` -> zero. Returns (series, label)."""
    import time
    for k in range(1, fred_attempts + 1):
        try:
            rf = build_historical_rf_daily(index, rf_source=rf_source,
                                           day_count=day_count, lag_days=lag_days)
            return rf, f"FRED:{rf_source.upper()}"
        except Exception as e:
            print(f"FRED risk-free attempt {k}/{fred_attempts} failed: {type(e).__name__}")
            time.sleep(1.0)
    try:
        rf = download_irx_rf_daily(index, day_count=day_count, lag_days=lag_days)
        print("FRED unavailable -> using Yahoo ^IRX (13-week T-bill) as the risk-free.")
        return rf, "Yahoo:^IRX"
    except Exception as e:
        print(f"Yahoo ^IRX fallback failed: {type(e).__name__}")
    import warnings as _w
    _w.warn("All risk-free sources failed; falling back to rf_daily = 0.0. "
            "Reported Sharpe/Sortino/Martin will use a zero risk-free rate.")
    rf = pd.Series(0.0, index=pd.DatetimeIndex(index), name="rf_daily")
    return rf, "ZERO"

In [ ]:
# ============================================================
# TRANSACTION-COST PROXIES
# ============================================================

def abdi_ranaldo_spread(high, low, close):
    high = pd.Series(high).astype(float)
    low = pd.Series(low).astype(float)
    close = pd.Series(close).astype(float)
    idx = high.dropna().index.intersection(low.dropna().index).intersection(close.dropna().index)
    high, low, close = high.loc[idx].sort_index(), low.loc[idx].sort_index(), close.loc[idx].sort_index()
    log_h, log_l, log_c = np.log(high), np.log(low), np.log(close)
    eta = 0.5 * (log_h + log_l)
    s2 = 4.0 * (log_c.shift(1) - eta.shift(1)) * (log_c.shift(1) - eta)
    spread = np.sqrt(np.maximum(s2, 0.0))
    spread.name = "ar_spread"
    return spread


def abdi_ranaldo_panel(px_high, px_low, px_close, tickers=None, clip_upper=0.10):
    if tickers is None:
        tickers = [c for c in px_high.columns if c in px_low.columns and c in px_close.columns]
    out = {}
    for t in tickers:
        h, l, c = px_high[t].dropna(), px_low[t].dropna(), px_close[t].dropna()
        idx = h.index.intersection(l.index).intersection(c.index)
        if len(idx) < 3:
            out[t] = pd.Series(dtype=float)
            continue
        s = abdi_ranaldo_spread(h.loc[idx], l.loc[idx], c.loc[idx])
        if clip_upper is not None:
            s = s.clip(upper=clip_upper)
        out[t] = s
    return pd.DataFrame(out).sort_index().reindex(columns=tickers)


def corwin_schultz_spread(high, low):
    """
    Corwin-Schultz (2012) FULL proportional spread from daily high/low.

    NOTE: the original paper also describes an *overnight adjustment* that shifts
    the previous day's high/low when the price gaps overnight, to strip the
    overnight drift out of the two-day range (gamma). That adjustment needs OPEN
    prices, which this OHLC bundle does not carry, so it is intentionally omitted.
    For liquid ETFs the bias from skipping it is small relative to the noise of
    the daily estimator, which is smoothed downstream with a trailing median.
    """
    high = pd.Series(high).astype(float)
    low = pd.Series(low).astype(float)
    idx = high.dropna().index.intersection(low.dropna().index)
    high, low = high.loc[idx].sort_index(), low.loc[idx].sort_index()
    hl = np.log(high / low)
    beta = hl.pow(2) + hl.shift(1).pow(2)
    high2 = pd.concat([high, high.shift(1)], axis=1).max(axis=1)
    low2 = pd.concat([low, low.shift(1)], axis=1).min(axis=1)
    gamma = np.log(high2 / low2).pow(2)
    k = 3 - 2 * np.sqrt(2)
    alpha = (np.sqrt(2 * beta) - np.sqrt(beta)) / k - np.sqrt(gamma / k)
    alpha = alpha.clip(lower=0)
    spread = 2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha))
    spread.name = "cs_spread"
    return spread


def corwin_schultz_panel(px_high, px_low, tickers=None, clip_upper=0.10):
    px_high, px_low = px_high.copy(), px_low.copy()
    if tickers is None:
        tickers = [c for c in px_high.columns if c in px_low.columns]
    out = {}
    for t in tickers:
        h, l = px_high[t].dropna(), px_low[t].dropna()
        idx = h.index.intersection(l.index)
        if len(idx) < 3:
            out[t] = pd.Series(dtype=float)
            continue
        s = corwin_schultz_spread(h.loc[idx], l.loc[idx])
        if clip_upper is not None:
            s = s.clip(upper=clip_upper)
        out[t] = s
    return pd.DataFrame(out).sort_index().reindex(columns=tickers)


def spread_panel_to_one_way_cost(spreads):
    return 0.5 * spreads.astype(float)


def constant_one_way_cost_panel(index, columns, one_way_bps):
    """
    Build a flat one-way transaction-cost panel of `one_way_bps` basis points.
    Useful as a sanity check against the noisy OHLC proxies (AR/CS), which tend
    to OVERstate spreads for liquid ETFs.
    """
    val = float(one_way_bps) / 1e4
    return pd.DataFrame(val, index=pd.DatetimeIndex(index), columns=list(columns), dtype=float)


def smooth_daily_cost_panel(cost_daily, lookback_days=21, agg="median"):
    cost_daily = cost_daily.sort_index().astype(float)
    min_obs = min(5, lookback_days)
    if agg == "median":
        return cost_daily.rolling(lookback_days, min_periods=min_obs).median()
    elif agg == "mean":
        return cost_daily.rolling(lookback_days, min_periods=min_obs).mean()
    raise ValueError("agg must be 'median' or 'mean'")


def align_costs_to_signal_dates(cost_daily, signal_dates, lookback_days=21, agg="median"):
    cost_daily = cost_daily.sort_index().astype(float)
    smoothed = smooth_daily_cost_panel(cost_daily, lookback_days=lookback_days, agg=agg)
    return smoothed.reindex(signal_dates)

In [ ]:
# ============================================================
# SIGNAL + WEIGHTS
# ============================================================

def month_end_trading_dates(idx):
    s = pd.Series(idx, index=idx)
    eom = s.groupby(s.index.to_period("M")).max()
    return pd.DatetimeIndex(eom.values)


def get_variant_params(variant):
    tdpm = 21
    if variant == "6-1":
        return 6 * tdpm, 1 * tdpm
    elif variant == "12-1":
        return 12 * tdpm, 1 * tdpm
    raise ValueError("variant must be '6-1' or '12-1'")


def cs_mom_signal_monthly(r, formation_days, skip_days):
    sig_daily = np.log1p(r).shift(skip_days).rolling(formation_days, min_periods=formation_days).sum()
    eom_dates = month_end_trading_dates(r.index)
    sig_m = sig_daily.loc[eom_dates].dropna(how="all")
    sig_m.index.name = "Date"
    return sig_m


def w_top1_80_20(sig_m):
    n = sig_m.shape[1]
    rest = 0.2 / (n - 1) if n > 1 else 0.0
    w = pd.DataFrame(rest, index=sig_m.index, columns=sig_m.columns, dtype=float)
    top = sig_m.idxmax(axis=1)
    for dt, a in top.items():
        w.loc[dt, a] = 0.8
    return w.div(w.sum(axis=1), axis=0)  # renormalise (no-op when n>1; handles n==1)


def w_top2_35_35(sig_m):
    n = sig_m.shape[1]
    rest = 0.3 / (n - 2) if n > 2 else 0.0
    w = pd.DataFrame(rest, index=sig_m.index, columns=sig_m.columns, dtype=float)
    for dt in sig_m.index:
        top2 = sig_m.loc[dt].nlargest(2).index
        w.loc[dt, top2] = 0.35
    return w.div(w.sum(axis=1), axis=0)  # renormalise (no-op when n>2; handles n<=2)


def w_equal(sig_m):
    n = sig_m.shape[1]
    return pd.DataFrame(1.0 / n, index=sig_m.index, columns=sig_m.columns, dtype=float)


# --- random structurally-identical weight builders (for the signal-permutation null) ---

def w_random_top1_80_20(sig_index, columns, rng):
    n = len(columns)
    rest = 0.2 / (n - 1) if n > 1 else 0.0
    m = len(sig_index)
    arr = np.full((m, n), rest, dtype=float)
    picks = rng.integers(0, n, size=m)
    arr[np.arange(m), picks] = 0.8
    arr = arr / arr.sum(axis=1, keepdims=True)
    return pd.DataFrame(arr, index=sig_index, columns=list(columns))


def w_random_top2_35_35(sig_index, columns, rng):
    n = len(columns)
    rest = 0.3 / (n - 2) if n > 2 else 0.0
    m = len(sig_index)
    arr = np.full((m, n), rest, dtype=float)
    for k in range(m):
        picks = rng.choice(n, size=min(2, n), replace=False)
        arr[k, picks] = 0.35
    arr = arr / arr.sum(axis=1, keepdims=True)
    return pd.DataFrame(arr, index=sig_index, columns=list(columns))

In [ ]:
# ============================================================
# BACKTEST
# ============================================================

def _entry_dates_from_signal_dates(trading_idx, signal_dates):
    trading_idx = pd.DatetimeIndex(trading_idx).sort_values().unique()
    signal_dates = pd.DatetimeIndex(signal_dates).sort_values().unique()
    entries, keep = [], []
    for dt in signal_dates:
        if dt < trading_idx[0]:
            continue
        pos = trading_idx.searchsorted(dt, side="right")
        if pos < len(trading_idx):
            entries.append(trading_idx[pos])
            keep.append(dt)
    return pd.DatetimeIndex(entries), pd.DatetimeIndex(keep)


def backtest_monthly_rebalance_with_drift(
    r, w_m,
    cost_oneway_daily=None,
    cost_lookback_days=21,
    cost_agg="median",
    charge_initial_trade=True,
    implementation_lag=1,
):
    """
    Monthly rebalance with intra-month drift and optional transaction costs.

    Timing (implementation_lag, in trading days):
      - signal observed at month-end close (signal date S);
      - the trade date is the NEXT trading day E (= first day after S);
      - the NEW target weights start earning returns `implementation_lag` days
        after the trade date.

      implementation_lag = 1 (default) => trade is executed at the close of E and
      the new weights first earn on E+1. On E itself the portfolio still holds the
      drifted PRE-trade weights. This is genuine next-day implementation and removes
      the ~1-day look-ahead present when new weights earn the trade-day return.

      implementation_lag = 0 reproduces the old "execute at the signal-date close"
      convention (new weights earn the E-day close-to-close return).
    """
    r = r.dropna(how="any").copy()
    entries, keep = _entry_dates_from_signal_dates(r.index, w_m.index)

    empty_s = pd.Series(dtype=float, name="portfolio")
    empty_w = pd.DataFrame(columns=r.columns, dtype=float)
    empty_info = pd.DataFrame(columns=["turnover", "tc_rate"], dtype=float)
    if len(entries) == 0:
        return empty_s, empty_s.copy(), empty_w, empty_w.copy(), empty_info

    w_sched = w_m.loc[keep].copy()
    w_sched.index = entries
    w_sched = w_sched[~w_sched.index.duplicated(keep="first")]
    entries = w_sched.index

    cost_sched = None
    if cost_oneway_daily is not None:
        cost_sig = align_costs_to_signal_dates(
            cost_oneway_daily.reindex(r.index), keep,
            lookback_days=cost_lookback_days, agg=cost_agg,
        )
        cost_sched = cost_sig.copy()
        cost_sched.index = entries
        cost_sched = cost_sched.reindex(index=entries, columns=r.columns).astype(float)

    n_days = len(r.index)
    base_pos = np.array([r.index.get_loc(dt) for dt in entries], dtype=int)
    eff_pos = base_pos + int(implementation_lag)

    valid = eff_pos < n_days
    keep_idx = np.where(valid)[0]
    eff_pos = eff_pos[valid]
    entries = entries[valid]
    w_sched = w_sched.iloc[keep_idx]
    if cost_sched is not None:
        cost_sched = cost_sched.iloc[keep_idx]
    if len(eff_pos) == 0:
        return empty_s, empty_s.copy(), empty_w, empty_w.copy(), empty_info

    port_gross = pd.Series(index=r.index, dtype=float, name="gross")
    port_net = pd.Series(index=r.index, dtype=float, name="net")
    w_d = pd.DataFrame(index=r.index, columns=r.columns, dtype=float)
    reb_info = pd.DataFrame(index=entries, columns=["turnover", "tc_rate"], dtype=float)

    current_w_pre = np.zeros(r.shape[1], dtype=float)
    rvals = r.values

    for i, start_pos in enumerate(eff_pos):
        end_pos = eff_pos[i + 1] - 1 if i + 1 < len(eff_pos) else n_days - 1

        target_w = w_sched.iloc[i].astype(float).values
        target_w = target_w / target_w.sum()

        if i == 0 and not charge_initial_trade:
            w_pre = target_w.copy()
        else:
            w_pre = current_w_pre.copy()

        turnover_rate = float(np.sum(np.abs(target_w - w_pre)))
        tc_rate = 0.0
        if cost_sched is not None:
            c_vec = np.nan_to_num(cost_sched.iloc[i].astype(float).values, nan=0.0)
            tc_rate = float(np.sum(c_vec * np.abs(target_w - w_pre)))
        reb_info.iloc[i] = [turnover_rate, tc_rate]

        current_w = target_w.copy()
        for pos in range(start_pos, end_pos + 1):
            x = rvals[pos]
            w_d.iloc[pos] = current_w
            gross_rp = float(np.dot(current_w, x))
            if pos == start_pos and tc_rate > 0:
                net_rp = (1.0 - tc_rate) * (1.0 + gross_rp) - 1.0
            else:
                net_rp = gross_rp
            port_gross.iloc[pos] = gross_rp
            port_net.iloc[pos] = net_rp
            current_w = current_w * (1.0 + x) / (1.0 + gross_rp)
        current_w_pre = current_w.copy()

    mask = port_gross.notna()
    return port_gross.loc[mask], port_net.loc[mask], w_d.loc[mask], w_sched, reb_info


def _turnover_summary_from_rebalance_info(reb_info):
    if reb_info.empty:
        return {k: np.nan for k in [
            "n_rebalances", "mean_turnover_two_way", "median_turnover_two_way",
            "mean_turnover_one_way", "annual_turnover_two_way_approx",
            "mean_tc_rate", "annual_tc_drag_approx", "total_tc_rate"]}
    n_reb = int(len(reb_info))
    mean_to = float(reb_info["turnover"].mean())
    med_to = float(reb_info["turnover"].median())
    mean_tc = float(reb_info["tc_rate"].mean())
    return {
        "n_rebalances": n_reb,
        "mean_turnover_two_way": mean_to,
        "median_turnover_two_way": med_to,
        "mean_turnover_one_way": 0.5 * mean_to,
        "annual_turnover_two_way_approx": 12.0 * mean_to,
        "mean_tc_rate": mean_tc,
        "annual_tc_drag_approx": 12.0 * mean_tc,
        "total_tc_rate": float(reb_info["tc_rate"].sum()),
    }


def run_factor_mom_from_returns(
    r, variant="6-1",
    cost_oneway_daily=None, cost_lookback_days=21, cost_agg="median",
    charge_initial_trade=True, implementation_lag=1,
):
    r_raw = r.copy()
    r = r.dropna(how="any").copy()

    formation_days, skip_days = get_variant_params(variant)
    sig_m = cs_mom_signal_monthly(r, formation_days=formation_days, skip_days=skip_days)

    weight_schemes = {
        "MomTop1_80_20": w_top1_80_20(sig_m),
        "MomTop2_35_35": w_top2_35_35(sig_m),
        "EW": w_equal(sig_m),
    }
    if cost_oneway_daily is not None:
        cost_oneway_daily = cost_oneway_daily.reindex(r.index)

    rets_gross, rets_net = {}, {}
    daily_weights, entry_weights, rebalance_info, turnover_summary = {}, {}, {}, {}

    for name, w_m in weight_schemes.items():
        p_gross, p_net, w_d, w_sched, reb_info = backtest_monthly_rebalance_with_drift(
            r, w_m, cost_oneway_daily=cost_oneway_daily,
            cost_lookback_days=cost_lookback_days, cost_agg=cost_agg,
            charge_initial_trade=charge_initial_trade, implementation_lag=implementation_lag,
        )
        rets_gross[name] = p_gross.rename(name)
        rets_net[name] = p_net.rename(name)
        daily_weights[name] = w_d
        entry_weights[name] = w_sched
        rebalance_info[name] = reb_info
        turnover_summary[name] = _turnover_summary_from_rebalance_info(reb_info)

    rets_gross = pd.concat(rets_gross.values(), axis=1)
    rets_net = pd.concat(rets_net.values(), axis=1)
    avg_daily_weights = pd.concat({k: v.mean() for k, v in daily_weights.items()}, axis=1)
    avg_entry_weights = pd.concat({k: v.mean() for k, v in entry_weights.items()}, axis=1)
    turnover_summary = pd.DataFrame(turnover_summary).T

    sample_info = {
        "raw_start": r_raw.index.min(), "raw_end": r_raw.index.max(),
        "common_start": r.index.min(), "common_end": r.index.max(),
        "raw_rows": int(len(r_raw)), "common_rows": int(len(r)),
    }
    return {
        "rets_gross": rets_gross, "rets_net": rets_net, "signal": sig_m,
        "daily_weights": daily_weights, "entry_weights": entry_weights,
        "avg_daily_weights": avg_daily_weights, "avg_entry_weights": avg_entry_weights,
        "rebalance_info": rebalance_info, "turnover_summary": turnover_summary,
        "sample_info": sample_info,
    }

In [ ]:
# ============================================================
# PERFORMANCE METRICS
# ============================================================

def max_drawdown_from_wealth(wealth):
    wealth = wealth.dropna().astype(float)
    if len(wealth) == 0:
        return np.nan
    peak = wealth.cummax()
    return float((wealth / peak - 1.0).min())


def ulcer_index(r):
    r = r.dropna().astype(float)
    if len(r) == 0:
        return np.nan
    wealth = (1.0 + r).cumprod()
    dd = wealth / wealth.cummax() - 1.0
    return float(np.sqrt(np.mean(dd ** 2)))


def cagr(r, ann=252):
    r = r.dropna().astype(float)
    if len(r) == 0:
        return np.nan
    wealth = (1.0 + r).cumprod()
    years = len(r) / ann
    if years <= 0:
        return np.nan
    w_last = wealth.iloc[-1]
    if not np.isfinite(w_last) or w_last <= 0:
        return np.nan
    return float(w_last ** (1 / years) - 1)


def _align_rf_to_returns(r, rf_daily=None):
    idx = r.dropna().index
    if rf_daily is None:
        return pd.Series(0.0, index=idx, name="rf_daily")
    if np.isscalar(rf_daily):
        return pd.Series(float(rf_daily), index=idx, name="rf_daily")
    if isinstance(rf_daily, pd.DataFrame):
        if rf_daily.shape[1] != 1:
            raise ValueError("rf_daily DataFrame must have exactly one column.")
        rf_daily = rf_daily.iloc[:, 0]
    rf = pd.Series(rf_daily).astype(float).sort_index().reindex(idx).ffill().bfill()
    rf.name = "rf_daily"
    return rf


def _annualized_rf_from_series(rf, ann=252):
    rf = pd.Series(rf).dropna().astype(float)
    if len(rf) == 0:
        return 0.0
    gross = (1.0 + rf).prod()
    years = len(rf) / ann
    return float(gross ** (1 / years) - 1.0) if years > 0 else 0.0


def perf_metrics(r, rf_daily=None, ann=252):
    r = r.dropna().astype(float)
    if len(r) < 5:
        return {"n": int(len(r))}
    rf = _align_rf_to_returns(r, rf_daily=rf_daily)
    ex = r.loc[rf.index] - rf

    mu_d = ex.mean()
    vol_d = ex.std(ddof=1)
    shr = (mu_d / vol_d) * np.sqrt(ann) if vol_d > 0 else np.nan

    # Sortino: target semideviation = RMS of downside excess returns (MAR = rf, i.e. 0 excess)
    neg = np.minimum(ex.values, 0.0)
    dvol = float(np.sqrt(np.mean(neg ** 2)))
    sor = (mu_d / dvol) * np.sqrt(ann) if dvol > 0 else np.nan

    cagr_ = cagr(r, ann=ann)
    ann_vol = r.std(ddof=1) * np.sqrt(ann)
    mxdd = max_drawdown_from_wealth((1.0 + r).cumprod())
    calmar = (cagr_ / abs(mxdd)) if (mxdd < 0 and np.isfinite(mxdd)) else np.nan
    ui = ulcer_index(r)
    rf_ann = _annualized_rf_from_series(rf, ann=ann)
    martin = ((cagr_ - rf_ann) / ui) if (ui > 0 and np.isfinite(ui)) else np.nan

    return {
        "n": int(len(r)), "CAGR": float(cagr_), "AnnVol": float(ann_vol),
        "MxDD": float(mxdd), "ShR": float(shr), "SoR": float(sor),
        "Calmar": float(calmar), "Ulcer": float(ui), "Martin": float(martin),
    }


def perf_table(F, rf_daily=None, ann=252):
    return pd.DataFrame({c: perf_metrics(F[c], rf_daily=rf_daily, ann=ann) for c in F.columns}).T


def wealth_index(F):
    return (1.0 + F).cumprod()

In [ ]:
# ============================================================
# STATIONARY BOOTSTRAP
# ============================================================

def stationary_bootstrap_indices(T, avg_block_len, rng):
    p = 1.0 / avg_block_len
    idx = np.empty(T, dtype=int)
    idx[0] = rng.integers(0, T)
    for t in range(1, T):
        if rng.random() < p:
            idx[t] = rng.integers(0, T)
        else:
            idx[t] = (idx[t - 1] + 1) % T
    return idx


def stationary_bootstrap_index_stream(T, avg_block_len, n_boot, seed):
    rng = np.random.default_rng(seed)
    for _ in range(n_boot):
        yield stationary_bootstrap_indices(T, avg_block_len, rng)


def summarize_vec(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return pd.Series({"mean": np.nan, "p05": np.nan, "p50": np.nan, "p95": np.nan, "n": 0})
    return pd.Series({
        "mean": float(np.mean(x)), "p05": float(np.quantile(x, 0.05)),
        "p50": float(np.quantile(x, 0.50)), "p95": float(np.quantile(x, 0.95)),
        "n": int(len(x)),
    })


def _metric_tables_from_vectors(mats, metrics, strategies):
    out = {}
    for m in metrics:
        rows = []
        for s in strategies:
            sm = summarize_vec(mats[(m, s)])
            sm["Strategy"] = s
            rows.append(sm)
        out[m] = pd.DataFrame(rows).set_index("Strategy")
    return out


# higher value = better, used for paired probabilities and null p-values
_HIGHER_IS_BETTER = {"CAGR": True, "AnnVol": False, "MxDD": True, "ShR": True,
                     "SoR": True, "Calmar": True, "Ulcer": False, "Martin": True}


def paired_difference_summary(mats, metrics, strat, bench):
    """Bootstrap distribution of (strat - bench), paired within each replica."""
    rows = {}
    for m in metrics:
        a = np.asarray(mats[(m, strat)], float)
        b = np.asarray(mats[(m, bench)], float)
        fin = np.isfinite(a) & np.isfinite(b)
        a, b = a[fin], b[fin]
        if len(a) == 0:
            rows[m] = {"mean_diff": np.nan, "p05": np.nan, "p50": np.nan,
                       "p95": np.nan, "P(strat>bench)": np.nan, "n": 0}
            continue
        d = a - b
        better = (a > b) if _HIGHER_IS_BETTER.get(m, True) else (a < b)
        rows[m] = {
            "mean_diff": float(np.mean(d)), "p05": float(np.quantile(d, 0.05)),
            "p50": float(np.quantile(d, 0.50)), "p95": float(np.quantile(d, 0.95)),
            "P(strat>bench)": float(np.mean(better)), "n": int(len(a)),
        }
    return pd.DataFrame(rows).T


def bootstrap_factor_mom_metrics(
    r, variant="6-1",
    cost_oneway_daily=None, cost_lookback_days=21, cost_agg="median",
    charge_initial_trade=True, implementation_lag=1,
    bench_returns=None, bench_name="SPY",
    metrics=("CAGR", "AnnVol", "ShR", "SoR", "Calmar", "Martin", "MxDD"),
    n_boot=1000, avg_block_len=10.0, seed=1, rf_daily=None, ann=252,
):
    """
    Stationary bootstrap on the COMMON sample (NaN rows dropped before resampling).
    The whole strategy is rebuilt inside every replica. Returns, the aligned cost
    panel and an optional benchmark series are resampled jointly (shared indices).
    """
    r = r.dropna(how="any").copy()                     # <-- common-sample fix
    if cost_oneway_daily is not None:
        cost_oneway_daily = cost_oneway_daily.reindex(r.index)
    if bench_returns is not None:
        bench_returns = pd.Series(bench_returns).reindex(r.index)

    actual = run_factor_mom_from_returns(
        r, variant=variant, cost_oneway_daily=cost_oneway_daily,
        cost_lookback_days=cost_lookback_days, cost_agg=cost_agg,
        charge_initial_trade=charge_initial_trade, implementation_lag=implementation_lag,
    )
    strategies = list(actual["rets_gross"].columns)
    if bench_returns is not None:
        strategies = strategies + [bench_name]

    actual_metrics_gross = perf_table(actual["rets_gross"], rf_daily=rf_daily, ann=ann)
    actual_metrics_net = perf_table(actual["rets_net"], rf_daily=rf_daily, ann=ann)
    strat_window = actual["rets_gross"].index   # post-warm-up window of the strategies
    if bench_returns is not None:
        # align the benchmark to the SAME window as the strategies for a fair head-to-head
        bench_aligned = bench_returns.reindex(strat_window)
        bm = pd.Series(perf_metrics(bench_aligned, rf_daily=rf_daily, ann=ann), name=bench_name)
        actual_metrics_gross = pd.concat([actual_metrics_gross, bm.to_frame().T])
        actual_metrics_net = pd.concat([actual_metrics_net, bm.to_frame().T])

    mats_gross = {(m, s): np.full(n_boot, np.nan) for m in metrics for s in strategies}
    mats_net = {(m, s): np.full(n_boot, np.nan) for m in metrics for s in strategies}

    T = len(r)
    for i, ii in enumerate(stationary_bootstrap_index_stream(T, avg_block_len, n_boot, seed)):
        sim_r = r.iloc[ii].copy(); sim_r.index = r.index
        sim_c = None
        if cost_oneway_daily is not None:
            sim_c = cost_oneway_daily.iloc[ii].copy(); sim_c.index = r.index
        sim_out = run_factor_mom_from_returns(
            sim_r, variant=variant, cost_oneway_daily=sim_c,
            cost_lookback_days=cost_lookback_days, cost_agg=cost_agg,
            charge_initial_trade=charge_initial_trade, implementation_lag=implementation_lag,
        )
        tab_gross = perf_table(sim_out["rets_gross"], rf_daily=rf_daily, ann=ann)
        tab_net = perf_table(sim_out["rets_net"], rf_daily=rf_daily, ann=ann)

        if bench_returns is not None:
            sim_b = bench_returns.iloc[ii].copy(); sim_b.index = r.index
            # restrict the benchmark to the same (post-warm-up) window as the strategies
            mb = perf_metrics(sim_b.reindex(sim_out["rets_gross"].index), rf_daily=rf_daily, ann=ann)

        for m in metrics:
            for s in actual["rets_gross"].columns:
                mats_gross[(m, s)][i] = tab_gross.loc[s, m] if m in tab_gross.columns else np.nan
                mats_net[(m, s)][i] = tab_net.loc[s, m] if m in tab_net.columns else np.nan
            if bench_returns is not None:
                v = mb.get(m, np.nan)
                mats_gross[(m, bench_name)][i] = v
                mats_net[(m, bench_name)][i] = v  # buy-and-hold: gross == net

    return {
        "actual_metrics_gross": actual_metrics_gross,
        "actual_metrics_net": actual_metrics_net,
        "bootstrap_tables_gross": _metric_tables_from_vectors(mats_gross, metrics, strategies),
        "bootstrap_tables_net": _metric_tables_from_vectors(mats_net, metrics, strategies),
        "simulated_metric_vectors_gross": mats_gross,
        "simulated_metric_vectors_net": mats_net,
        "strategies": strategies,
    }

In [ ]:
# ============================================================
# RANDOM-SELECTION (SIGNAL-PERMUTATION) NULL
# ============================================================

def random_selection_null(
    r, sig_m, weight_builder,
    cost_oneway_daily=None, cost_lookback_days=21, cost_agg="median",
    charge_initial_trade=True, implementation_lag=1,
    metrics=("CAGR", "ShR", "SoR", "Calmar", "Martin", "MxDD"),
    use_net=True, n_random=1000, seed=12345, rf_daily=None, ann=252,
):
    """
    Build a null of strategies that are STRUCTURALLY IDENTICAL to the real one
    (same rebalance dates, same concentration profile, same costs, same timing)
    but pick the held factor(s) AT RANDOM each month. Any edge of the real
    strategy over this null is attributable to the momentum SIGNAL, not to the
    static factor exposure of a particular benchmark blend.
    """
    r = r.dropna(how="any").copy()
    if cost_oneway_daily is not None:
        cost_oneway_daily = cost_oneway_daily.reindex(r.index)
    rng = np.random.default_rng(seed)
    cols = list(sig_m.columns)
    out = {m: np.full(n_random, np.nan) for m in metrics}
    for j in range(n_random):
        w_rand = weight_builder(sig_m.index, cols, rng)
        p_gross, p_net, *_ = backtest_monthly_rebalance_with_drift(
            r, w_rand, cost_oneway_daily=cost_oneway_daily,
            cost_lookback_days=cost_lookback_days, cost_agg=cost_agg,
            charge_initial_trade=charge_initial_trade, implementation_lag=implementation_lag,
        )
        series = p_net if (use_net and cost_oneway_daily is not None) else p_gross
        mm = perf_metrics(series, rf_daily=rf_daily, ann=ann)
        for m in metrics:
            out[m][j] = mm.get(m, np.nan)
    return out


def null_pvalues(actual_metrics_row, null_dist, metrics):
    rows = {}
    for m in metrics:
        x = np.asarray(null_dist[m], float)
        x = x[np.isfinite(x)]
        a = float(actual_metrics_row[m])
        if len(x) == 0:
            rows[m] = {"actual": a, "null_mean": np.nan, "null_p05": np.nan,
                       "null_p95": np.nan, "pval_one_sided": np.nan, "n": 0}
            continue
        if _HIGHER_IS_BETTER.get(m, True):
            p = float(np.mean(x >= a))   # fraction of random strategies that do at least as well
        else:
            p = float(np.mean(x <= a))
        rows[m] = {
            "actual": a, "null_mean": float(np.mean(x)),
            "null_p05": float(np.quantile(x, 0.05)), "null_p95": float(np.quantile(x, 0.95)),
            "pval_one_sided": p, "n": int(len(x)),
        }
    return pd.DataFrame(rows).T

In [ ]:
# ============================================================
# ACF BLOCK-LENGTH HEURISTIC
# ============================================================

def _acf(x, nlags):
    x = np.asarray(x, float)
    x = x - np.mean(x)
    denom = np.dot(x, x)
    if denom <= 0:
        return np.zeros(nlags + 1)
    out = np.empty(nlags + 1, float)
    out[0] = 1.0
    for k in range(1, nlags + 1):
        out[k] = np.dot(x[:-k], x[k:]) / denom
    return out


def choose_block_length_by_acf_matching(
    s, candidates=(5, 10, 20, 30), nlags=20, n_boot=300,
    use_abs=True, seed=1, distance="weighted",
):
    r = s.dropna().astype(float)
    x = np.abs(r.values) if use_abs else r.values
    target = _acf(x, nlags)
    rng = np.random.default_rng(seed)
    scores, boot_means = {}, {}
    for L in candidates:
        acfs = np.zeros((n_boot, nlags + 1))
        for b in range(n_boot):
            ii = stationary_bootstrap_indices(len(x), float(L), rng)
            acfs[b] = _acf(x[ii], nlags)
        m = acfs.mean(axis=0)
        boot_means[L] = m
        diff = m[1:] - target[1:]
        if distance == "l1":
            score = float(np.mean(np.abs(diff)))
        elif distance == "l2":
            score = float(np.sqrt(np.mean(diff ** 2)))
        elif distance == "weighted":
            w = 1.0 / np.arange(1, nlags + 1)
            score = float(np.sqrt(np.mean(diff ** 2 * w)))
        else:
            raise ValueError("distance must be 'l1', 'l2', or 'weighted'")
        scores[L] = score
    scores = pd.Series(scores).sort_index()
    return {"best_L": int(scores.idxmin()), "scores": scores,
            "target_acf": target, "boot_acf_mean": boot_means}

In [ ]:
# ============================================================
# ALTERNATIVE WEIGHTING (risk-based: inverse-volatility)
# ============================================================

def _trailing_vol_at_signals(r, sig_index, lookback=63):
    """Daily stdev over a trailing window, sampled at the signal dates."""
    vol = r.rolling(lookback, min_periods=max(20, lookback // 3)).std()
    return vol.reindex(sig_index).bfill()


def w_topk_inverse_vol(sig_m, r, k=1, lookback=63):
    """
    Select the top-k factors by momentum and weight them by INVERSE VOLATILITY
    (equal-risk proxy), with zero in the rest. Removes the arbitrary 80/20 (or
    35/35) concentration so that *selection* is separated from *concentration*.
    """
    vol = _trailing_vol_at_signals(r, sig_m.index, lookback=lookback)
    arr = np.zeros((len(sig_m.index), sig_m.shape[1]), dtype=float)
    cols = list(sig_m.columns)
    for j, dt in enumerate(sig_m.index):
        top = sig_m.loc[dt].nlargest(k).index
        iv = 1.0 / vol.loc[dt, top].replace(0.0, np.nan)
        iv = iv / iv.sum()
        for name, wv in iv.items():
            arr[j, cols.index(name)] = float(wv)
    w = pd.DataFrame(arr, index=sig_m.index, columns=cols)
    return w.div(w.sum(axis=1), axis=0)


def w_inverse_vol_all(sig_m, r, lookback=63):
    """Inverse-volatility across ALL factors: a more neutral (risk-parity-like)
    benchmark than equal weight, which ignores the very different factor vols."""
    vol = _trailing_vol_at_signals(r, sig_m.index, lookback=lookback)
    iv = 1.0 / vol.reindex(sig_m.index)
    w = iv.div(iv.sum(axis=1), axis=0)
    return w.reindex(columns=sig_m.columns).fillna(0.0)


def run_named_strategy(r, weight_matrix, cost_oneway_daily=None, cost_lookback_days=21,
                       cost_agg="median", charge_initial_trade=True, implementation_lag=1):
    """Backtest a single arbitrary monthly weight matrix; returns (gross, net) series."""
    g, n, _, _, _ = backtest_monthly_rebalance_with_drift(
        r, weight_matrix, cost_oneway_daily=cost_oneway_daily,
        cost_lookback_days=cost_lookback_days, cost_agg=cost_agg,
        charge_initial_trade=charge_initial_trade, implementation_lag=implementation_lag)
    return g, n


# ============================================================
# SERIAL-DEPENDENCE DIAGNOSTICS (Ljung-Box, ARCH-LM)
# ============================================================

def ljung_box(x, lags=10):
    from scipy import stats
    x = np.asarray(pd.Series(x).dropna(), float)
    x = x - x.mean()
    T = len(x)
    denom = np.dot(x, x)
    Q = 0.0
    for k in range(1, lags + 1):
        rho = np.dot(x[:-k], x[k:]) / denom
        Q += rho * rho / (T - k)
    Q *= T * (T + 2)
    p = float(stats.chi2.sf(Q, lags))
    return {"stat": float(Q), "lags": int(lags), "pval": p}


def arch_lm(x, lags=10):
    """Engle's ARCH-LM test for conditional heteroskedasticity."""
    from scipy import stats
    e = np.asarray(pd.Series(x).dropna(), float)
    e = e - e.mean()
    e2 = e ** 2
    T = len(e2)
    Y = e2[lags:]
    X = np.column_stack([np.ones(T - lags)] + [e2[lags - j: T - j] for j in range(1, lags + 1)])
    b, *_ = np.linalg.lstsq(X, Y, rcond=None)
    resid = Y - X @ b
    ss_tot = np.sum((Y - Y.mean()) ** 2)
    r2 = 1.0 - np.sum(resid ** 2) / ss_tot if ss_tot > 0 else 0.0
    lm = (T - lags) * r2
    p = float(stats.chi2.sf(lm, lags))
    return {"stat": float(lm), "lags": int(lags), "pval": p}


def dependence_diagnostics(series_dict, lags=10):
    """Table of Ljung-Box (levels & squares) and ARCH-LM p-values per series."""
    rows = {}
    for name, s in series_dict.items():
        s = pd.Series(s).dropna()
        lb = ljung_box(s, lags)
        lb2 = ljung_box(s ** 2, lags)
        al = arch_lm(s, lags)
        rows[name] = {
            "LB_levels_stat": lb["stat"], "LB_levels_p": lb["pval"],
            "LB_squares_stat": lb2["stat"], "LB_squares_p": lb2["pval"],
            "ARCH_LM_stat": al["stat"], "ARCH_LM_p": al["pval"], "lags": lags,
        }
    return pd.DataFrame(rows).T


# ============================================================
# SPANNING REGRESSION (OLS + Newey-West HAC)
# ============================================================

def newey_west_ols(y, X, lags=None):
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    T, k = X.shape
    XtX = X.T @ X
    XtX_inv = np.linalg.inv(XtX)
    b = XtX_inv @ (X.T @ y)
    e = y - X @ b
    if lags is None:
        lags = int(np.floor(4 * (T / 100.0) ** (2.0 / 9.0)))
    Xe = X * e[:, None]
    S = Xe.T @ Xe
    for l in range(1, lags + 1):
        w = 1.0 - l / (lags + 1.0)
        G = Xe[l:].T @ Xe[:-l]
        S += w * (G + G.T)
    cov = XtX_inv @ S @ XtX_inv
    se = np.sqrt(np.diag(cov))
    tstat = b / se
    ss_res = float(e @ e)
    ss_tot = float(((y - y.mean()) ** 2).sum())
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return {"beta": b, "se": se, "t": tstat, "r2": r2, "lags": lags, "n": T, "cov": cov}


def spanning_regression(strat_returns, factor_returns, rf_daily=None,
                        extra_returns=None, lags=None, ann=252):
    """
    Regress the strategy's EXCESS return on the factor ETFs' excess returns (and any
    extra benchmark, e.g. SPY). A Newey-West alpha that is statistically zero means
    the strategy is 'spanned' by static factor exposure -- the timing/selection adds
    nothing beyond a fixed combination of the factors.
    """
    from scipy import stats
    s = pd.Series(strat_returns).dropna()
    F = pd.DataFrame(factor_returns).reindex(s.index)
    if extra_returns is not None:
        extra = pd.DataFrame(extra_returns)
        if isinstance(extra_returns, pd.Series):
            extra = extra_returns.to_frame()
        F = pd.concat([F, extra.reindex(s.index)], axis=1)
    idx = s.index.intersection(F.dropna().index)
    s, F = s.loc[idx], F.loc[idx]
    rf = _align_rf_to_returns(s, rf_daily=rf_daily)
    y = (s - rf).values
    Xf = (F.sub(rf, axis=0)).values
    X = np.column_stack([np.ones(len(y)), Xf])
    res = newey_west_ols(y, X, lags=lags)
    names = ["alpha"] + list(F.columns)
    coefs = pd.DataFrame({"coef": res["beta"], "se": res["se"], "t": res["t"]}, index=names)
    coefs["pval"] = 2 * stats.norm.sf(np.abs(coefs["t"]))
    out = {
        "alpha_daily": float(res["beta"][0]),
        "alpha_annual": float(res["beta"][0] * ann),
        "alpha_t": float(res["t"][0]),
        "alpha_p": float(coefs.loc["alpha", "pval"]),
        "r2": float(res["r2"]), "n": int(res["n"]), "nw_lags": int(res["lags"]),
        "coefficients": coefs,
    }
    return out


# ============================================================
# SHARPE-RATIO INFERENCE (Ledoit-Wolf 2008; PSR / Deflated Sharpe)
# ============================================================

def _bartlett_lrv(Y, bw):
    T = Y.shape[0]
    Yc = Y - Y.mean(axis=0, keepdims=True)
    Omega = (Yc.T @ Yc) / T
    for l in range(1, bw + 1):
        w = 1.0 - l / (bw + 1.0)
        G = (Yc[l:].T @ Yc[:-l]) / T
        Omega += w * (G + G.T)
    return Omega


def _andrews_bw_bartlett(Y):
    """Andrews (1991) automatic bandwidth for the Bartlett kernel via AR(1) plug-in."""
    T = Y.shape[0]
    num = 0.0
    den = 0.0
    for j in range(Y.shape[1]):
        x = Y[:, j] - Y[:, j].mean()
        x0, x1 = x[:-1], x[1:]
        rho = float(np.dot(x0, x1) / np.dot(x0, x0)) if np.dot(x0, x0) > 0 else 0.0
        rho = np.clip(rho, -0.97, 0.97)
        s2 = np.var(x, ddof=0)
        num += 4 * rho ** 2 * s2 ** 2 / ((1 - rho) ** 6 * (1 + rho) ** 2)
        den += s2 ** 2 / (1 - rho) ** 4
    alpha1 = num / den if den > 0 else 0.0
    bw = 1.1447 * (alpha1 * T) ** (1.0 / 3.0)
    return max(1, int(np.ceil(bw)))


def _sharpe_diff_and_se(r1, r2, bw=None):
    r1 = np.asarray(r1, float); r2 = np.asarray(r2, float)
    mu1, mu2 = r1.mean(), r2.mean()
    g1, g2 = (r1 ** 2).mean(), (r2 ** 2).mean()
    v1, v2 = g1 - mu1 ** 2, g2 - mu2 ** 2
    sr1, sr2 = mu1 / np.sqrt(v1), mu2 / np.sqrt(v2)
    delta = sr1 - sr2
    grad = np.array([
        g1 / v1 ** 1.5,
        -g2 / v2 ** 1.5,
        -0.5 * mu1 / v1 ** 1.5,
        0.5 * mu2 / v2 ** 1.5,
    ])
    Y = np.column_stack([r1, r2, r1 ** 2, r2 ** 2])
    if bw is None:
        bw = _andrews_bw_bartlett(Y)
    Omega = _bartlett_lrv(Y, bw)
    var_delta = float(grad @ Omega @ grad) / len(r1)
    se = np.sqrt(var_delta) if var_delta > 0 else np.nan
    return delta, se, bw


def circular_block_indices(T, block_size, rng):
    block_size = max(1, int(block_size))
    n_blocks = int(np.ceil(T / block_size))
    starts = rng.integers(0, T, size=n_blocks)
    idx = np.concatenate([(np.arange(s, s + block_size) % T) for s in starts])[:T]
    return idx


def sharpe_diff_test(r1, r2, block_size=None, n_boot=999, seed=0, ann=252):
    """
    Ledoit-Wolf (2008) test for the difference of two Sharpe ratios under non-iid
    returns: HAC standard error (Bartlett kernel, Andrews bandwidth) + studentized
    circular-block bootstrap p-value. Pass returns in the SAME units (e.g. both
    excess). Two-sided H0: SR1 == SR2.
    """
    s1 = pd.Series(r1).dropna()
    s2 = pd.Series(r2).dropna()
    idx = s1.index.intersection(s2.index)
    a = s1.loc[idx].values; b = s2.loc[idx].values
    T = len(a)
    delta, se, bw = _sharpe_diff_and_se(a, b)
    stat = delta / se if (se and np.isfinite(se)) else np.nan
    if block_size is None:
        block_size = max(1, bw)
    rng = np.random.default_rng(seed)
    count = 0
    valid = 0
    for _ in range(n_boot):
        ii = circular_block_indices(T, block_size, rng)
        db, seb, _ = _sharpe_diff_and_se(a[ii], b[ii], bw=bw)
        if seb and np.isfinite(seb):
            valid += 1
            if abs((db - delta) / seb) >= abs(stat):
                count += 1
    pval = (count + 1) / (valid + 1) if valid > 0 else np.nan
    return {
        "sharpe1_ann": float(a.mean() / a.std(ddof=0) * np.sqrt(ann)),
        "sharpe2_ann": float(b.mean() / b.std(ddof=0) * np.sqrt(ann)),
        "delta_daily": float(delta), "delta_ann": float(delta * np.sqrt(ann)),
        "se_daily": float(se), "tstat": float(stat), "pval": float(pval),
        "block_size": int(block_size), "hac_bw": int(bw), "n": int(T), "n_boot": int(valid),
    }


def probabilistic_sharpe_ratio(returns, sr_benchmark=0.0, ann=252):
    """PSR: probability the true (non-annualised) Sharpe exceeds a benchmark,
    accounting for skewness and kurtosis of the returns (Bailey & Lopez de Prado)."""
    from scipy import stats
    r = np.asarray(pd.Series(returns).dropna(), float)
    T = len(r)
    mu, sd = r.mean(), r.std(ddof=1)
    sr = mu / sd
    g3 = stats.skew(r, bias=False)
    g4 = stats.kurtosis(r, fisher=False, bias=False)  # non-excess kurtosis
    denom = np.sqrt(1 - g3 * sr + (g4 - 1) / 4.0 * sr ** 2)
    z = (sr - sr_benchmark) * np.sqrt(T - 1) / denom
    return {"sharpe_per_obs": float(sr), "sharpe_ann": float(sr * np.sqrt(ann)),
            "skew": float(g3), "kurtosis": float(g4),
            "psr": float(stats.norm.cdf(z)), "T": int(T)}


def deflated_sharpe_ratio(returns, sr_trials, ann=252):
    """
    Deflated Sharpe Ratio (Bailey & Lopez de Prado 2014): PSR evaluated against the
    Sharpe expected from the BEST of N independent trials, where N and the dispersion
    of trial Sharpes come from the specification search. `sr_trials` = per-observation
    Sharpe ratios of all specifications tried.
    """
    from scipy import stats
    sr_trials = np.asarray([s for s in sr_trials if np.isfinite(s)], float)
    N = len(sr_trials)
    var_sr = np.var(sr_trials, ddof=1) if N > 1 else 0.0
    gamma = 0.5772156649
    e1 = stats.norm.ppf(1 - 1.0 / N) if N > 1 else 0.0
    e2 = stats.norm.ppf(1 - 1.0 / (N * np.e)) if N > 1 else 0.0
    sr0 = np.sqrt(var_sr) * ((1 - gamma) * e1 + gamma * e2)
    psr = probabilistic_sharpe_ratio(returns, sr_benchmark=sr0, ann=ann)
    return {"n_trials": int(N), "sr0_threshold_per_obs": float(sr0),
            "sr0_threshold_ann": float(sr0 * np.sqrt(ann)),
            "dsr": float(psr["psr"]), "sharpe_ann": float(psr["sharpe_ann"]),
            "trial_sharpe_std_per_obs": float(np.sqrt(var_sr))}


# ============================================================
# BREAK-EVEN TRANSACTION COST
# ============================================================

def _sharpe_ann(series, rf_daily=None, ann=252):
    s = pd.Series(series).dropna()
    rf = _align_rf_to_returns(s, rf_daily=rf_daily)
    ex = (s - rf)
    sd = ex.std(ddof=1)
    return float(ex.mean() / sd * np.sqrt(ann)) if sd > 0 else np.nan


def breakeven_cost_bps(r, variant="6-1", strat="MomTop1_80_20", mode="cagr_vs_ew",
                       bench_returns=None, rf_daily=None, ann=252, c_max_bps=100.0,
                       implementation_lag=1, cost_lookback_days=21, cost_agg="median",
                       charge_initial_trade=True):
    """
    One-way cost (bps) that drives the strategy's edge to zero. Modes:
      'cagr_vs_ew'      : CAGR(strat) - CAGR(EW), both charged the same flat cost
      'sharpe_vs_ew'    : Sharpe(strat) - Sharpe(EW), same flat cost on both
      'cagr_vs_spy'     : CAGR(strat at cost) - CAGR(SPY buy&hold)
      'excess_cagr_rf'  : CAGR(strat at cost) - annualised risk-free
    """
    from scipy.optimize import brentq
    r_common = r.dropna(how="any")
    cols = list(r_common.columns)

    def run_at(c_bps):
        cp = constant_one_way_cost_panel(r_common.index, cols, c_bps) if c_bps > 0 else None
        return run_factor_mom_from_returns(
            r, variant=variant, cost_oneway_daily=cp, cost_lookback_days=cost_lookback_days,
            cost_agg=cost_agg, charge_initial_trade=charge_initial_trade,
            implementation_lag=implementation_lag)

    def g(c_bps):
        out = run_at(c_bps)
        s = out["rets_net"][strat]
        if mode == "cagr_vs_ew":
            return cagr(s, ann) - cagr(out["rets_net"]["EW"], ann)
        if mode == "sharpe_vs_ew":
            return _sharpe_ann(s, rf_daily, ann) - _sharpe_ann(out["rets_net"]["EW"], rf_daily, ann)
        if mode == "cagr_vs_spy":
            if bench_returns is None:
                raise ValueError("bench_returns required for cagr_vs_spy")
            bench = pd.Series(bench_returns).reindex(s.index).dropna()
            return cagr(s.reindex(bench.index), ann) - cagr(bench, ann)
        if mode == "excess_cagr_rf":
            rf = _align_rf_to_returns(s, rf_daily=rf_daily)
            return cagr(s, ann) - _annualized_rf_from_series(rf, ann)
        raise ValueError("unknown mode")

    g0 = g(0.0)
    if not np.isfinite(g0) or g0 <= 0:
        return {"breakeven_bps": 0.0, "edge_at_zero": float(g0), "mode": mode,
                "note": "no positive edge even at zero cost"}
    gmax = g(c_max_bps)
    if gmax > 0:
        return {"breakeven_bps": float("inf"), "edge_at_zero": float(g0),
                "edge_at_cmax": float(gmax), "c_max_bps": c_max_bps, "mode": mode,
                "note": f"edge survives beyond {c_max_bps} bps one-way"}
    c_star = brentq(g, 0.0, c_max_bps, xtol=1e-3)
    return {"breakeven_bps": float(c_star), "edge_at_zero": float(g0), "mode": mode}


# ============================================================
# AUTOMATIC BLOCK LENGTH (Politis-White 2004 / PPW 2009)
# ============================================================

def optimal_block_length_ppw(series):
    """Patton-Politis-White optimal block length (via the `arch` implementation)."""
    from arch.bootstrap import optimal_block_length
    s = pd.Series(series).dropna()
    ob = optimal_block_length(s.values)
    if isinstance(ob, pd.DataFrame):
        row = ob.iloc[0]
        return {"stationary": float(row.get("stationary", np.nan)),
                "circular": float(row.get("circular", np.nan))}
    return {"stationary": float(ob[0]), "circular": float(ob[1])}


# ============================================================
# BCa INTERVALS (P&L circular-block bootstrap + block jackknife)
# ============================================================

def _metric_sharpe(series, ann=252):
    s = np.asarray(series, float)
    sd = s.std(ddof=1)
    return float(s.mean() / sd * np.sqrt(ann)) if sd > 0 else np.nan


def _metric_cagr(series, ann=252):
    return cagr(pd.Series(series), ann=ann)


def block_bootstrap_pnl_dist(series, metric_fn, block_size, n_boot=1000, seed=0):
    s = np.asarray(pd.Series(series).dropna(), float)
    T = len(s)
    rng = np.random.default_rng(seed)
    out = np.empty(n_boot)
    for b in range(n_boot):
        ii = circular_block_indices(T, block_size, rng)
        out[b] = metric_fn(s[ii])
    return out


def _block_jackknife_values(series, metric_fn, block_size):
    s = np.asarray(pd.Series(series).dropna(), float)
    T = len(s)
    nb = int(np.ceil(T / block_size))
    vals = []
    for k in range(nb):
        lo, hi = k * block_size, min((k + 1) * block_size, T)
        keep = np.concatenate([s[:lo], s[hi:]])
        if len(keep) > 5:
            vals.append(metric_fn(keep))
    return np.asarray(vals, float)


def bca_interval(series, metric_fn, block_size, n_boot=1000, alpha=0.05, seed=0):
    """BCa confidence interval for a return-series functional, using a circular-block
    P&L bootstrap and a delete-block jackknife (note: this resamples the realised P&L,
    distinct from the strategy-rebuild bootstrap)."""
    from scipy import stats
    theta = metric_fn(np.asarray(pd.Series(series).dropna(), float))
    boot = block_bootstrap_pnl_dist(series, metric_fn, block_size, n_boot=n_boot, seed=seed)
    boot = boot[np.isfinite(boot)]
    jack = _block_jackknife_values(series, metric_fn, block_size)
    z0 = stats.norm.ppf(np.mean(boot < theta)) if 0 < np.mean(boot < theta) < 1 else 0.0
    jbar = jack.mean()
    num = np.sum((jbar - jack) ** 3)
    den = 6.0 * (np.sum((jbar - jack) ** 2) ** 1.5)
    a = num / den if den != 0 else 0.0
    zl, zu = stats.norm.ppf(alpha / 2), stats.norm.ppf(1 - alpha / 2)
    def adj(z):
        return stats.norm.cdf(z0 + (z0 + z) / (1 - a * (z0 + z)))
    lo, hi = adj(zl), adj(zu)
    return {"theta": float(theta),
            "lo": float(np.quantile(boot, lo)), "hi": float(np.quantile(boot, hi)),
            "alpha": alpha, "z0": float(z0), "acceleration": float(a),
            "block_size": int(block_size), "n_boot": int(len(boot))}

In [ ]:
# ============================================================
# REGIME / SUBSAMPLE STABILITY
# ============================================================

def metrics_by_calendar_year(series, rf_daily=None, ann=252):
    s = pd.Series(series).dropna()
    rows = {}
    for y, grp in s.groupby(s.index.year):
        if len(grp) >= 20:
            rf = None if rf_daily is None else pd.Series(rf_daily).reindex(grp.index)
            rows[int(y)] = perf_metrics(grp, rf_daily=rf, ann=ann)
    return pd.DataFrame(rows).T


def metrics_two_halves(series, rf_daily=None, ann=252):
    s = pd.Series(series).dropna()
    mid = len(s) // 2
    out = {}
    for label, sub in [("first_half", s.iloc[:mid]), ("second_half", s.iloc[mid:])]:
        rf = None if rf_daily is None else pd.Series(rf_daily).reindex(sub.index)
        out[label] = perf_metrics(sub, rf_daily=rf, ann=ann)
    return pd.DataFrame(out).T


def metrics_by_vol_regime(series, market_returns, lookback=21, n_regimes=2,
                          rf_daily=None, ann=252):
    """Split performance by the trailing realised volatility of the market series
    (terciles/halves). No extra data needed beyond the benchmark."""
    s = pd.Series(series).dropna()
    mkt = pd.Series(market_returns).reindex(s.index)
    rv = mkt.rolling(lookback, min_periods=lookback // 2).std()
    labels = (["low", "high"] if n_regimes == 2 else ["low", "mid", "high"])
    q = pd.qcut(rv.reindex(s.index), n_regimes, labels=labels)
    out = {}
    for lab in labels:
        sub = s[q == lab]
        if len(sub) >= 20:
            rf = None if rf_daily is None else pd.Series(rf_daily).reindex(sub.index)
            out[f"{lab}_vol"] = perf_metrics(sub, rf_daily=rf, ann=ann)
    return pd.DataFrame(out).T


# ============================================================
# SOPHISTICATED BOOTSTRAP ENGINES (FHS AR-GJR-GARCH; VAR-sieve)
# ============================================================

def _fit_argjr_garch(series, ar_lags=1):
    """Fit AR(p)-GJR-GARCH(1,1) (Gaussian QMLE) on a single return series.
    Returns mean/vol params, standardized residuals and the unconditional variance.
    Internals work in PERCENT for numerical stability."""
    from arch.univariate import arch_model
    y = pd.Series(series).dropna().astype(float) * 100.0
    mean = "AR" if ar_lags and ar_lags > 0 else "Constant"
    am = arch_model(y, mean=mean, lags=ar_lags if ar_lags else 0,
                    vol="GARCH", p=1, o=1, q=1, dist="normal", rescale=False)
    res = am.fit(disp="off", show_warning=False)
    p = res.params
    const = float(p.get("Const", 0.0))
    phi = np.array([float(p.get(f"y[{j}]", p.get(f"{y.name}[{j}]", 0.0)))
                    for j in range(1, (ar_lags or 0) + 1)], dtype=float)
    omega = float(p["omega"]); alpha = float(p["alpha[1]"])
    gamma = float(p.get("gamma[1]", 0.0)); beta = float(p["beta[1]"])
    z = res.std_resid.dropna().values
    persist = alpha + beta + 0.5 * gamma
    uncond_var = omega / (1 - persist) if 0 < persist < 1 else float(np.var(res.resid.dropna()))
    return {"const": const, "phi": phi, "omega": omega, "alpha": alpha,
            "gamma": gamma, "beta": beta, "z": z, "uncond_var": uncond_var,
            "p": int(ar_lags or 0)}


def _reinflate_argjr(fit, z_path, burnin=300):
    """Re-inflate one series from a path of standardized residuals via the AR-GJR
    recursion. z_path length must be >= T_target + burnin; returns last T_target
    simple returns (de-scaled from percent)."""
    const, phi = fit["const"], fit["phi"]
    omega, alpha, gamma, beta = fit["omega"], fit["alpha"], fit["gamma"], fit["beta"]
    p = fit["p"]
    n = len(z_path)
    h = np.empty(n); eps = np.empty(n); y = np.empty(n)
    h0 = max(fit["uncond_var"], 1e-12)
    h_cap = 1e4 * h0          # cap conditional variance to stop explosive runaway -> inf/nan
    mu0 = const / (1 - phi.sum()) if (p and abs(1 - phi.sum()) > 1e-6) else const
    prev_y = np.full(max(p, 1), mu0)
    eps_prev = 0.0; h_prev = h0
    for t in range(n):
        h[t] = omega + (alpha + (gamma if eps_prev < 0 else 0.0)) * eps_prev ** 2 + beta * h_prev
        if not np.isfinite(h[t]) or h[t] > h_cap:
            h[t] = h_cap
        eps[t] = np.sqrt(max(h[t], 1e-16)) * z_path[t]
        mu = const + (np.dot(phi, prev_y[:p][::-1]) if p else 0.0)
        y[t] = mu + eps[t]
        if p:
            prev_y = np.concatenate([prev_y[1:], [y[t]]])
        eps_prev, h_prev = eps[t], h[t]
    # sanitize then clip simulated simple returns to [-99%, +100%]: the AR-GJR recursion with
    # fat-tailed residuals (and near-unit estimated persistence) can occasionally draw an extreme
    # day that would make (1+r) <= 0 (breaking log1p/CAGR) or overflow cumprod. Non-binding on
    # real ETF data; only trims pathological simulated draws.
    out = np.nan_to_num(y[burnin:] / 100.0, nan=0.0, posinf=1.0, neginf=-0.99)
    return np.clip(out, -0.99, 1.0)


def fhs_simulate_panel(R, ar_lags=1, resid_block=None, n_sims=500, seed=1, burnin=300):
    """
    Filtered Historical Simulation for a panel. Each column is filtered with its own
    AR(p)-GJR-GARCH(1,1); the matrix of standardized residuals is resampled BY ROW
    (preserving contemporaneous cross-sectional correlation) and re-inflated through
    each column's recursion. `resid_block`=None -> iid rows; int -> circular blocks.
    Yields simulated panels (same shape/columns as R).
    """
    R = R.dropna(how="any")
    cols = list(R.columns)
    T = len(R)
    fits = {c: _fit_argjr_garch(R[c], ar_lags=ar_lags) for c in cols}
    Z = np.column_stack([fits[c]["z"][-(min(len(fits[c]["z"]) for c in cols)):] for c in cols])
    nZ = Z.shape[0]
    rng = np.random.default_rng(seed)
    need = T + burnin
    for _ in range(n_sims):
        if resid_block is None:
            ridx = rng.integers(0, nZ, size=need)
        else:
            ridx = circular_block_indices(nZ, resid_block, rng)
            while len(ridx) < need:
                ridx = np.concatenate([ridx, circular_block_indices(nZ, resid_block, rng)])
            ridx = ridx[:need]
        sim = {c: _reinflate_argjr(fits[c], Z[ridx, j], burnin=burnin)
               for j, c in enumerate(cols)}
        yield pd.DataFrame(sim, index=R.index, columns=cols)


def _fit_var(R, p=1):
    Y = R.values
    T, k = Y.shape
    Xrows = []
    for t in range(p, T):
        lagvec = np.concatenate([Y[t - j] for j in range(1, p + 1)])
        Xrows.append(np.concatenate([[1.0], lagvec]))
    X = np.array(Xrows)
    Yt = Y[p:]
    B, *_ = np.linalg.lstsq(X, Yt, rcond=None)   # ( (1+kp) x k )
    resid = Yt - X @ B
    return {"B": B, "resid": resid, "p": p, "k": k, "Y0": Y[:p].copy(), "index": R.index, "cols": list(R.columns)}


def _var_spectral_radius(B, p, k):
    A = [B[1 + j * k: 1 + (j + 1) * k, :].T for j in range(p)]   # each k x k
    top = np.hstack(A)
    if p == 1:
        comp = top
    else:
        comp = np.vstack([top, np.hstack([np.eye(k * (p - 1)), np.zeros((k * (p - 1), k))])])
    return float(np.max(np.abs(np.linalg.eigvals(comp))))


def var_sieve_simulate_panel(R, p=1, n_sims=500, seed=1):
    """VAR(p) sieve with a recursive-design WILD bootstrap (Rademacher), robust to
    conditional heteroskedasticity; preserves contemporaneous cross-sectional
    correlation (whole residual rows are sign-flipped together)."""
    R = R.dropna(how="any")
    fit = _fit_var(R, p=p)
    B, resid, k, cols = fit["B"], fit["resid"], fit["k"], fit["cols"]
    radius = _var_spectral_radius(B, p, k)
    if radius >= 1.0:
        scale = 0.99 / radius
        for j in range(p):
            B[1 + j * k: 1 + (j + 1) * k, :] *= scale
    T = len(R); nE = resid.shape[0]
    rng = np.random.default_rng(seed)
    for _ in range(n_sims):
        w = rng.choice([-1.0, 1.0], size=nE)
        estar = resid * w[:, None]
        Y = np.empty((T, k))
        Y[:p] = fit["Y0"]
        for t in range(p, T):
            lagvec = np.concatenate([Y[t - j] for j in range(1, p + 1)])
            x = np.concatenate([[1.0], lagvec])
            Y[t] = x @ B + estar[t - p]
        yield pd.DataFrame(Y, index=R.index, columns=cols)


def bootstrap_metrics_via_engine(
    r_factors, engine="fhs", variant="6-1", bench_returns=None, bench_name="SPY",
    flat_cost_oneway_bps=None, ar_lags=1, var_lags=1, resid_block=None,
    metrics=("CAGR", "AnnVol", "ShR", "SoR", "Calmar", "Martin", "MxDD"),
    n_sims=500, seed=1, rf_daily=None, ann=252, implementation_lag=1,
    cost_lookback_days=21, cost_agg="median", charge_initial_trade=True):
    """
    Dependence-aware bootstrap that GENERATES new panels with a chosen engine
    ('fhs' or 'var_sieve'), rebuilding the whole strategy inside each replica.
    Returns / benchmark are simulated jointly. Because simulated panels carry no
    OHLC, costs use a flat one-way bps panel (`flat_cost_oneway_bps`), or gross if None.
    Output mirrors bootstrap_factor_mom_metrics so paired_difference_summary applies.
    """
    rf = r_factors.dropna(how="any")
    cols = list(rf.columns)
    if bench_returns is not None:
        b = pd.Series(bench_returns).reindex(rf.index)
        R = pd.concat([rf, b.rename(bench_name)], axis=1).dropna(how="any")
    else:
        R = rf.copy()
    fac_cols = cols
    strategies = ["MomTop1_80_20", "MomTop2_35_35", "EW"] + ([bench_name] if bench_returns is not None else [])

    if engine == "fhs":
        gen = fhs_simulate_panel(R, ar_lags=ar_lags, resid_block=resid_block, n_sims=n_sims, seed=seed)
    elif engine == "var_sieve":
        gen = var_sieve_simulate_panel(R, p=var_lags, n_sims=n_sims, seed=seed)
    else:
        raise ValueError("engine must be 'fhs' or 'var_sieve'")

    mats_net = {(m, s): np.full(n_sims, np.nan) for m in metrics for s in strategies}
    for i, sim_full in enumerate(gen):
        sim_fac = sim_full[fac_cols]
        cp = (constant_one_way_cost_panel(sim_fac.index, fac_cols, flat_cost_oneway_bps)
              if flat_cost_oneway_bps else None)
        sim_out = run_factor_mom_from_returns(
            sim_fac, variant=variant, cost_oneway_daily=cp,
            cost_lookback_days=cost_lookback_days, cost_agg=cost_agg,
            charge_initial_trade=charge_initial_trade, implementation_lag=implementation_lag)
        tab = perf_table(sim_out["rets_net"], rf_daily=rf_daily, ann=ann)
        if bench_returns is not None:
            mb = perf_metrics(sim_full[bench_name], rf_daily=rf_daily, ann=ann)
        for m in metrics:
            for s in ["MomTop1_80_20", "MomTop2_35_35", "EW"]:
                mats_net[(m, s)][i] = tab.loc[s, m] if m in tab.columns else np.nan
            if bench_returns is not None:
                mats_net[(m, bench_name)][i] = mb.get(m, np.nan)
    return {
        "engine": engine,
        "bootstrap_tables_net": _metric_tables_from_vectors(mats_net, metrics, strategies),
        "simulated_metric_vectors_net": mats_net,
        "strategies": strategies, "n_sims": int(n_sims),
    }


# ============================================================
# MULTIPLE TESTING UNDER SPECIFICATION SEARCH
# ============================================================

def build_spec_return_matrix(r, variants=("6-1", "12-1"),
                             concentrations=("top1", "top2"),
                             cost_panels=None, rf_daily=None, implementation_lag=1,
                             cost_lookback_days=21, cost_agg="median",
                             charge_initial_trade=True, ann=252):
    """
    Net daily returns for every specification in the grid
    (variant x concentration x cost-estimator). Returns (matrix, sharpe_trials)
    where columns are 'variant|conc|cost' and sharpe_trials are the per-observation
    Sharpe ratios of each spec (input to the Deflated Sharpe Ratio).
    """
    if cost_panels is None:
        cost_panels = {"GROSS": None}
    conc_col = {"top1": "MomTop1_80_20", "top2": "MomTop2_35_35"}
    cols = {}
    sharpe_trials = {}
    for v in variants:
        for cname, cpanel in cost_panels.items():
            out = run_factor_mom_from_returns(
                r, variant=v, cost_oneway_daily=cpanel, cost_lookback_days=cost_lookback_days,
                cost_agg=cost_agg, charge_initial_trade=charge_initial_trade,
                implementation_lag=implementation_lag)
            for conc in concentrations:
                key = f"{v}|{conc}|{cname}"
                s = out["rets_net"][conc_col[conc]]
                cols[key] = s
                rf = _align_rf_to_returns(s, rf_daily=rf_daily)
                ex = (s - rf)
                sd = ex.std(ddof=1)
                sharpe_trials[key] = float(ex.mean() / sd) if sd > 0 else np.nan
    mat = pd.concat(cols, axis=1)
    mat.columns = list(cols.keys())
    return mat, sharpe_trials


def reality_check_white(spec_matrix, bench_returns, block_size=10, n_boot=1000, seed=0):
    """
    White's (2000) Reality Check. H0: the best specification does not outperform the
    benchmark once the full search is accounted for. Works on excess-over-benchmark
    daily returns; circular-block bootstrap, recentred max statistic.
    """
    M = pd.DataFrame(spec_matrix)
    b = pd.Series(bench_returns).reindex(M.index)
    idx = M.dropna().index.intersection(b.dropna().index)
    D = M.loc[idx].sub(b.loc[idx], axis=0).values
    T, K = D.shape
    fbar = D.mean(axis=0)
    V = np.sqrt(T) * np.max(fbar)
    rng = np.random.default_rng(seed)
    Vstar = np.empty(n_boot)
    for bnum in range(n_boot):
        ii = circular_block_indices(T, block_size, rng)
        fstar = D[ii].mean(axis=0)
        Vstar[bnum] = np.sqrt(T) * np.max(fstar - fbar)
    p = float((np.sum(Vstar >= V) + 1) / (n_boot + 1))
    best = M.columns[int(np.argmax(fbar))]
    return {"pvalue": p, "stat": float(V), "best_spec": best,
            "mean_excess_ann": pd.Series(fbar * 252, index=M.columns).sort_values(ascending=False),
            "block_size": int(block_size), "n_boot": int(n_boot), "n": int(T), "K": int(K)}


def spa_test_hansen(spec_matrix, bench_returns, block_size=10, reps=1000, seed=0):
    """
    Hansen's (2005) Superior Predictive Ability test via the `arch` implementation.
    Returns lower / consistent / upper p-values (consistent is recommended; the
    upper bound is the conservative White-Reality-Check-style configuration).
    Operates on losses = -returns, benchmark = the EW (or chosen) series.
    """
    from arch.bootstrap import SPA
    M = pd.DataFrame(spec_matrix)
    b = pd.Series(bench_returns).reindex(M.index)
    idx = M.dropna().index.intersection(b.dropna().index)
    losses_models = (-M.loc[idx]).values
    losses_bench = (-b.loc[idx]).values
    spa = SPA(losses_bench, losses_models, block_size=block_size, reps=reps,
              bootstrap="circular", seed=seed)
    spa.compute()
    pv = spa.pvalues
    return {"pvalues": {k: float(pv[k]) for k in pv.index},
            "block_size": int(block_size), "reps": int(reps), "n": int(len(idx)),
            "K": int(M.shape[1])}

In [ ]:
# ============================================================
# EXTENSIONS 2: long-short factors, walk-forward, quantile/rank weights, conditional spanning, static replication, selection mechanism
# ============================================================

def download_famafrench_factors(start="2000-01-01"):
    """Daily LONG-SHORT academic factor returns from Kenneth French's data library via
    pandas_datareader: Size=SMB, Value=HML, Quality=RMW, Investment=CMA, Momentum=Mom.
    Returns (factor_rets [decimal], rf_daily [decimal], mkt_excess [decimal]).
    Requires network access to French's library (mba.tuck.dartmouth.edu)."""
    from pandas_datareader import data as _pdr
    ff5 = _pdr.DataReader("F-F_Research_Data_5_Factors_2x3_daily", "famafrench", start=start)[0]
    mom = _pdr.DataReader("F-F_Momentum_Factor_daily", "famafrench", start=start)[0]
    ff5 = ff5.copy(); ff5.columns = [c.strip() for c in ff5.columns]
    mom = mom.copy(); mom.columns = [c.strip() for c in mom.columns]
    mom_col = [c for c in mom.columns if c.lower().startswith("mom")][0]
    df = ff5.join(mom, how="inner") / 100.0
    df.index = pd.to_datetime(df.index)
    rets = pd.DataFrame({"Size": df["SMB"], "Value": df["HML"], "Quality": df["RMW"],
                         "Investment": df["CMA"], "Momentum": df[mom_col]}).dropna()
    return rets, df["RF"].reindex(rets.index).rename("rf_daily"), df["Mkt-RF"].reindex(rets.index).rename("Mkt-RF")


def load_factor_returns_csv(path, rf_col=None, mkt_col=None):
    """Fallback loader: a CSV of daily factor returns (decimal) indexed by date."""
    df = pd.read_csv(path, index_col=0, parse_dates=True).sort_index()
    rf = df.pop(rf_col).rename("rf_daily") if (rf_col and rf_col in df.columns) else None
    mkt = df.pop(mkt_col).rename("Mkt-RF") if (mkt_col and mkt_col in df.columns) else None
    return df.astype(float), rf, mkt


def backtest_returns_panel_monthly(rets, w_m, implementation_lag=1, cost_oneway_bps=0.0):
    """Monthly-rebalanced portfolio on a RETURN panel (e.g. long-short factor returns):
    FIXED weights within each month (no price drift, which is the right convention for
    zero-investment factor returns), next-day implementation via `implementation_lag`,
    optional flat one-way cost charged additively on |Delta w| at each rebalance."""
    rets = rets.dropna(how="any").copy()
    entries, keep = _entry_dates_from_signal_dates(rets.index, w_m.index)
    if len(entries) == 0:
        return pd.Series(dtype=float, name="port")
    w_sched = w_m.loc[keep].copy(); w_sched.index = entries
    w_sched = w_sched[~w_sched.index.duplicated(keep="first")]; entries = w_sched.index
    n_days = len(rets.index)
    base = np.array([rets.index.get_loc(dt) for dt in entries], dtype=int)
    eff = base + int(implementation_lag)
    valid = eff < n_days
    eff = eff[valid]; w_sched = w_sched.iloc[np.where(valid)[0]]
    if len(eff) == 0:
        return pd.Series(dtype=float, name="port")
    port = pd.Series(index=rets.index, dtype=float, name="port")
    R = rets.values
    c = float(cost_oneway_bps) / 1e4
    prev_w = np.zeros(rets.shape[1])
    for i, s in enumerate(eff):
        e = eff[i + 1] - 1 if i + 1 < len(eff) else n_days - 1
        w = w_sched.iloc[i].astype(float).values
        cost = c * float(np.sum(np.abs(w - prev_w))) if c > 0 else 0.0
        for pos in range(s, e + 1):
            rp = float(np.dot(w, R[pos]))
            if pos == s and cost > 0:
                rp -= cost
            port.iloc[pos] = rp
        prev_w = w
    return port.loc[port.notna()]


def w_quantile_long(sig_m, top_frac=0.4):
    """Long-only, equal-weight the TOP fraction of factors by momentum (rank/quantile
    portfolio). For a large universe this is the natural construction; for N=5 with
    top_frac=0.4 it holds the top 2. Separates selection from fixed 80/20 concentration."""
    n = sig_m.shape[1]; k = max(1, int(round(top_frac * n)))
    cols = list(sig_m.columns); arr = np.zeros((len(sig_m), n))
    for j, dt in enumerate(sig_m.index):
        for c in sig_m.loc[dt].nlargest(k).index:
            arr[j, cols.index(c)] = 1.0 / k
    return pd.DataFrame(arr, index=sig_m.index, columns=cols)


def w_rank_proportional(sig_m):
    """Long-only weights proportional to the cross-sectional momentum RANK (1..N)."""
    r = sig_m.rank(axis=1, method="average")
    return r.div(r.sum(axis=1), axis=0)


def run_factor_mom_on_panel(rets, variant="6-1", implementation_lag=1, cost_oneway_bps=0.0,
                            weighting="concentrated"):
    """Cross-sectional factor momentum on a generic return panel (used for the long-short
    academic factors). Returns {'rets': DataFrame of strategy returns, 'signal': sig_m}."""
    rets = rets.dropna(how="any").copy()
    fd, sd = get_variant_params(variant)
    sig = cs_mom_signal_monthly(rets, fd, sd)
    if weighting == "concentrated":
        schemes = {"MomTop1": w_top1_80_20(sig), "MomTop2": w_top2_35_35(sig), "EW": w_equal(sig)}
    elif weighting == "quantile":
        schemes = {"MomQ_top40": w_quantile_long(sig, 0.4), "MomRank": w_rank_proportional(sig),
                   "EW": w_equal(sig)}
    else:
        raise ValueError("weighting must be 'concentrated' or 'quantile'")
    out = {k: backtest_returns_panel_monthly(rets, w, implementation_lag, cost_oneway_bps).rename(k)
           for k, w in schemes.items()}
    return {"rets": pd.concat(out.values(), axis=1), "signal": sig}


def walk_forward_select(spec_matrix, min_train_days=504, reselect_every_days=63,
                        rf_daily=None, ann=252):
    """True out-of-sample walk-forward: expanding window, re-pick the best specification by
    in-sample Sharpe every `reselect_every_days`, then bank that spec's OOS returns. The
    spec returns are causal, so selecting on the past and banking the future is valid. This
    answers the spec-search-overfit worry DIRECTLY (not just via the SPA p-value)."""
    M = pd.DataFrame(spec_matrix).dropna(how="any")
    idx = M.index; T = len(idx)
    oos = pd.Series(index=idx, dtype=float, name="OOS_selected")
    chosen = []
    t = int(min_train_days)
    while t < T:
        train = M.iloc[:t]
        rf = _align_rf_to_returns(train.iloc[:, 0], rf_daily=rf_daily).reindex(train.index)
        sh = {}
        for col in M.columns:
            ex = train[col] - rf
            sd = ex.std(ddof=1)
            sh[col] = float(ex.mean() / sd) if sd > 0 else -np.inf
        best = max(sh, key=sh.get)
        end = min(t + int(reselect_every_days), T)
        oos.iloc[t:end] = M[best].iloc[t:end].values
        chosen.append({"date": idx[t], "spec": best, "is_sharpe_ann": sh[best] * np.sqrt(ann)})
        t = end
    oos = oos.dropna()
    return {"oos_returns": oos, "chosen": pd.DataFrame(chosen).set_index("date"),
            "oos_metrics": perf_metrics(oos, rf_daily=rf_daily, ann=ann),
            "n_reselections": len(chosen)}


def vol_regime_dummy(market_returns, index, lookback=21, q=0.5):
    """1 = high-volatility day, by trailing realised vol of `market_returns` above its
    q-quantile; aligned to `index`."""
    m = pd.Series(market_returns)
    rv = m.rolling(lookback, min_periods=max(2, lookback // 2)).std()
    thr = rv.quantile(q)
    d = (rv > thr).astype(float).reindex(pd.DatetimeIndex(index)).ffill().bfill()
    return d.rename("high_vol")


def spanning_regression_conditional(strat, factors, regime, rf_daily=None, extra=None,
                                    lags=None, ann=252):
    """Conditional spanning: two alphas (low- and high-regime) via full interaction of the
    constant and factor loadings with a regime dummy, Newey-West HAC. Tests whether the
    zero full-sample alpha hides a non-zero alpha within a regime."""
    from scipy import stats
    s = pd.Series(strat).dropna()
    F = pd.DataFrame(factors).reindex(s.index)
    if extra is not None:
        ex = extra.to_frame() if isinstance(extra, pd.Series) else pd.DataFrame(extra)
        F = pd.concat([F, ex.reindex(s.index)], axis=1)
    D = pd.Series(regime).reindex(s.index).astype(float)
    idx = s.index.intersection(F.dropna().index).intersection(D.dropna().index)
    s, F, D = s.loc[idx], F.loc[idx], D.loc[idx]
    rf = _align_rf_to_returns(s, rf_daily=rf_daily)
    y = (s - rf).values
    Fx = F.sub(rf, axis=0).values
    Dv = D.values
    X = np.column_stack([np.ones(len(y)), Dv, Fx, Fx * Dv[:, None]])
    res = newey_west_ols(y, X, lags=lags)
    b, cov = res["beta"], res["cov"]
    a_low = b[0]; se_low = np.sqrt(cov[0, 0])
    a_high = b[0] + b[1]; se_high = np.sqrt(cov[0, 0] + cov[1, 1] + 2 * cov[0, 1])
    def _row(a, se):
        tt = a / se if se > 0 else np.nan
        return {"alpha_ann": float(a * ann), "t": float(tt), "p": float(2 * stats.norm.sf(abs(tt)))}
    return {"low_regime": _row(a_low, se_low), "high_regime": _row(a_high, se_high),
            "delta_high_minus_low": {"alpha_ann": float(b[1] * ann),
                                     "t": float(b[1] / np.sqrt(cov[1, 1]) if cov[1, 1] > 0 else np.nan)},
            "n": int(res["n"]), "nw_lags": int(res["lags"]),
            "n_high": int(D.sum()), "n_low": int((1 - D).sum())}


def rolling_spanning_alpha(strat, factors, rf_daily=None, window=504, extra=None, ann=252):
    """Rolling annualised OLS alpha of the strategy vs the factors (+ extra). Lets you SEE
    whether the alpha drifts / concentrates in episodes rather than being a single number."""
    s = pd.Series(strat).dropna()
    F = pd.DataFrame(factors).reindex(s.index)
    if extra is not None:
        ex = extra.to_frame() if isinstance(extra, pd.Series) else pd.DataFrame(extra)
        F = pd.concat([F, ex.reindex(s.index)], axis=1)
    idx = s.index.intersection(F.dropna().index)
    s, F = s.loc[idx], F.loc[idx]
    rf = _align_rf_to_returns(s, rf_daily=rf_daily)
    y = (s - rf).values
    X = np.column_stack([np.ones(len(y)), F.sub(rf, axis=0).values])
    out = {}
    for i in range(window, len(y) + 1):
        yy = y[i - window:i]; XX = X[i - window:i]
        bb, *_ = np.linalg.lstsq(XX, yy, rcond=None)
        out[idx[i - 1]] = float(bb[0] * ann)
    return pd.Series(out, name="rolling_alpha_ann")


def static_replication(strat, factors, rf_daily=None, extra=None, ann=252):
    """Show the strategy is a FIXED factor blend: fit the spanning betas, rebuild the static
    beta-weighted factor portfolio (alpha excluded) and report how well it tracks the
    strategy, plus the implied long-only weights for display."""
    sp = spanning_regression(strat, factors, rf_daily=rf_daily, extra_returns=extra)
    betas = sp["coefficients"]["coef"].drop("alpha")
    s = pd.Series(strat).dropna()
    F = pd.DataFrame(factors).reindex(s.index)
    if extra is not None:
        ex = extra.to_frame() if isinstance(extra, pd.Series) else pd.DataFrame(extra)
        F = pd.concat([F, ex.reindex(s.index)], axis=1)
    idx = s.index.intersection(F.dropna().index)
    s, F = s.loc[idx], F.loc[idx]
    rf = _align_rf_to_returns(s, rf_daily=rf_daily)
    Fx = F.sub(rf, axis=0)
    fitted = Fx[betas.index].mul(betas, axis=1).sum(axis=1)
    actual_ex = (s - rf).reindex(fitted.index)
    corr = float(np.corrcoef(fitted.values, actual_ex.values)[0, 1])
    wlong = betas.clip(lower=0)
    wlong = (wlong / wlong.sum()) if wlong.sum() > 0 else wlong
    return {"betas": betas, "implied_long_weights": wlong, "corr_static_vs_actual": corr,
            "r2": float(sp["r2"]), "alpha_ann": float(sp["alpha_annual"]),
            "static_sharpe_ann": _sharpe_ann(fitted, rf_daily=0.0, ann=ann),
            "actual_sharpe_ann": _sharpe_ann(actual_ex, rf_daily=0.0, ann=ann)}


def selection_quality_by_regime(sig_m, factor_rets, regime=None, k=1):
    """Mechanism for the positive finding: for each month, rank the forward holding-period
    return of the momentum-PICKED factor(s) among all factors (1=worst..N=best) and record
    whether the pick was the worst/best. Aggregated overall and by regime, vs the random
    benchmark. If momentum AVOIDS the worst factor (esp. in high-vol months), that explains
    why it beats the random null on drawdowns but not on the mean."""
    rets = factor_rets.dropna(how="any")
    entries, keep = _entry_dates_from_signal_dates(rets.index, sig_m.index)
    N = rets.shape[1]
    rows = []
    for i in range(len(entries)):
        e0 = rets.index.get_loc(entries[i])
        e1 = rets.index.get_loc(entries[i + 1]) if i + 1 < len(entries) else len(rets.index)
        if e1 <= e0:
            continue
        cum = (1 + rets.iloc[e0:e1]).prod() - 1
        ranks = cum.rank()  # 1=worst .. N=best
        pick = sig_m.loc[keep[i]].nlargest(k).index
        reg = np.nan if regime is None else float(pd.Series(regime).reindex([entries[i]]).ffill().iloc[0])
        rows.append({"date": entries[i], "pick_rank": float(ranks[pick].mean()),
                     "picked_worst": bool(cum.idxmin() in pick), "picked_best": bool(cum.idxmax() in pick),
                     "regime": reg})
    df = pd.DataFrame(rows)
    def _agg(d):
        return {"mean_rank": float(d["pick_rank"].mean()), "P_worst": float(d["picked_worst"].mean()),
                "P_best": float(d["picked_best"].mean()), "months": int(len(d))}
    summary = {"overall": _agg(df)}
    summary["overall"].update({"random_mean_rank": (N + 1) / 2, "random_P_worst": 1.0 / N})
    if regime is not None and df["regime"].notna().any():
        for lab, val in [("low_vol", 0.0), ("high_vol", 1.0)]:
            sub = df[df["regime"] == val]
            if len(sub):
                summary[lab] = _agg(sub)
    return {"per_month": df, "summary": pd.DataFrame(summary).T}


def universe_robustness(rets, cost_oneway_daily=None, rf_daily=None, variant="6-1",
                        implementation_lag=1, cost_lookback_days=21, cost_agg="median",
                        charge_initial_trade=True, ann=252,
                        strat="MomTop1_80_20", bench="EW",
                        n_grid=None, n_random_subsets=15, subset_seed=0):
    """Sensitivity of the factor-momentum result to the CHOICE OF UNIVERSE (addresses the
    researcher-degrees-of-freedom worry) and the BREADTH GRADIENT.
      (a) leave-one-out: drop each factor in turn, recompute the net Sharpe of `strat`, of
          `bench` and their gap. If the gap survives dropping any single ETF, it is not driven
          by one (possibly dubious) name.
      (b) size sweep: for each N, draw random subsets of N factors and summarise the net Sharpe
          of `strat` and the strat-minus-bench gap. A gap rising with N is direct evidence the
          effect is breadth-driven.
    Reuses the ETF drift backtest with costs (run_factor_mom_from_returns)."""
    import itertools
    rets = rets.dropna(how="any").copy()
    cols_all = list(rets.columns)
    K = len(cols_all)
    if cost_oneway_daily is not None:
        cost_oneway_daily = cost_oneway_daily.reindex(rets.index)

    def _sharpes(cols):
        cols = list(cols)
        cp = cost_oneway_daily[cols] if cost_oneway_daily is not None else None
        out = run_factor_mom_from_returns(
            rets[cols], variant=variant, cost_oneway_daily=cp,
            cost_lookback_days=cost_lookback_days, cost_agg=cost_agg,
            charge_initial_trade=charge_initial_trade, implementation_lag=implementation_lag)
        t = perf_table(out["rets_net"], rf_daily=rf_daily, ann=ann)
        ss = float(t.loc[strat, "ShR"]) if (strat in t.index and "ShR" in t.columns) else np.nan
        sb = float(t.loc[bench, "ShR"]) if (bench in t.index and "ShR" in t.columns) else np.nan
        return ss, sb

    full_s, full_b = _sharpes(cols_all)
    loo = {}
    for c in cols_all:
        ss, sb = _sharpes([x for x in cols_all if x != c])
        loo[f"drop_{c}"] = {"sharpe_strat": ss, "sharpe_bench": sb, "gap": ss - sb}
    loo_df = pd.DataFrame(loo).T
    loo_df.loc["__FULL__"] = {"sharpe_strat": full_s, "sharpe_bench": full_b, "gap": full_s - full_b}

    if n_grid is None:
        n_grid = list(range(max(2, K - 6), K + 1))
    rng = np.random.default_rng(subset_seed)
    sweep = {}
    for N in n_grid:
        if N >= K:
            subsets = [tuple(cols_all)]
        else:
            all_comb = list(itertools.combinations(cols_all, N))
            if len(all_comb) <= n_random_subsets:
                subsets = all_comb
            else:
                pick = rng.choice(len(all_comb), size=n_random_subsets, replace=False)
                subsets = [all_comb[i] for i in pick]
        ss_list, gaps = [], []
        for cols in subsets:
            ss, sb = _sharpes(cols)
            ss_list.append(ss); gaps.append(ss - sb)
        gaps = np.asarray(gaps, float); ss_arr = np.asarray(ss_list, float)
        sweep[N] = {"n_subsets": len(subsets),
                    "sharpe_strat_mean": float(np.nanmean(ss_arr)),
                    "gap_mean": float(np.nanmean(gaps)),
                    "gap_p05": float(np.nanpercentile(gaps, 5)),
                    "gap_p95": float(np.nanpercentile(gaps, 95)),
                    "gap_frac_pos": float(np.mean(gaps > 0))}
    sweep_df = pd.DataFrame(sweep).T
    sweep_df.index.name = "N_factors"
    return {"leave_one_out": loo_df, "size_sweep": sweep_df,
            "full_sharpe_strat": full_s, "full_sharpe_bench": full_b}

## Configuration

- `DATA_SOURCE`: `"hdf"` reproduces the original master-project panel; `"yahoo"` is the public-data workflow.
- `IMPLEMENTATION_LAG`: `1` = trade at the close of the day after the signal (next-day); `0` = trade at the signal-date close (legacy).
- Transaction costs: smoothed one-way half-spread proxies (AR/CS) or a flat `TC_FIXED_BPS`, applied at monthly rebalances.
- `RUN_BENCHMARK` adds `SPY` as a passive market benchmark.

In [ ]:

# ============================================================
# USER CONFIGURATION
# ============================================================

DATA_SOURCE = "yahoo"           # "yahoo" or "hdf"
HDF_PATH = None                 # e.g. "data/factor_etfs.h5" if DATA_SOURCE == "hdf"
HDF_KEY = "/FactorETFs"
START_DATE = "2000-01-01"
VARIANT = "6-1"                 # "6-1" or "12-1"

# --- Implementation timing ---
# 1 = genuine next-day implementation: trade at the close of the trading day AFTER
#     the signal date; new weights first earn on the following day. This removes the
#     ~1-day look-ahead that exists when new weights earn the trade-day return.
# 0 = legacy "execute at the signal-date close" convention.
IMPLEMENTATION_LAG = 1

# --- Historical risk-free rate ---
USE_HISTORICAL_RF = True
RF_SOURCE = "DGS3MO"            # "DGS3MO", "DTB3", or "SOFR" (SOFR only from 2018-04-03)
RF_DAY_COUNT = 360
RF_LAG_DAYS = 1

# --- Market benchmark ---
RUN_BENCHMARK = True
BENCH_TICKER = "SPY"

# --- Transaction costs ---
RUN_TRANSACTION_COSTS = True
TC_ESTIMATOR = "AR"             # primary estimator for plots/tables: "AR", "CS" or "FIXED"
RUN_TC_ESTIMATOR_COMPARISON = True
TC_ESTIMATORS_TO_COMPARE = ("AR", "CS", "FIXED")   # FIXED = flat bps sanity check
TC_FIXED_BPS = 2.0              # one-way cost in bps for the FIXED estimator
TC_LOOKBACK_DAYS = 21
TC_AGG = "median"               # "median" or "mean"
TC_SPREAD_CLIP_UPPER = 0.10
TC_CHARGE_INITIAL_TRADE = True

# --- Bootstrap ---
RUN_ACF_BLOCK_SELECTION = True
ACF_CANDIDATES = (5, 10, 20, 30)
ACF_NLAGS = 20
ACF_N_BOOT = 300

RUN_BOOTSTRAP = True
N_BOOT = 1000
AVG_BLOCK_LEN = 10              # used when RUN_ACF_BLOCK_SELECTION = False
BOOTSTRAP_SEED = 7

# --- Random-selection (signal-permutation) null ---
RUN_RANDOM_NULL = True
N_RANDOM = 1000
RANDOM_NULL_SEED = 12345

# metrics used for the paired-difference / engine tables
PAIRED_METRICS = ("CAGR", "ShR", "SoR", "Calmar", "Martin", "MxDD")

# --- Alternative weighting (separate selection from concentration) ---
RUN_ALT_WEIGHTING = True
ALT_WEIGHT_LOOKBACK = 63

# --- Serial-dependence diagnostics (motivate the bootstrap engine) ---
RUN_DEPENDENCE_DIAGNOSTICS = True
DIAG_LAGS = 10

# --- Spanning regression (Newey-West alpha vs the factor ETFs + SPY) ---
RUN_SPANNING = True

# --- Sharpe-ratio inference (Ledoit-Wolf delta-Sharpe; PSR) ---
RUN_SHARPE_INFERENCE = True
LW_N_BOOT = 4999

# --- Break-even one-way transaction cost ---
RUN_BREAKEVEN = True
BREAKEVEN_C_MAX_BPS = 200.0

# --- BCa intervals on the realized P&L ---
RUN_BCA = True
BCA_N_BOOT = 2000

# --- Regime / subsample stability ---
RUN_REGIME = True
REGIME_VOL_LOOKBACK = 21
REGIME_N = 2

# --- Dependence-aware bootstrap engines (FHS AR-GJR-GARCH; VAR-sieve) ---
RUN_ENGINE_BOOTSTRAP = True
ENGINES = ("fhs", "var_sieve")     # subset of these
ENGINE_N_SIMS = 500
ENGINE_AR_LAGS = 1
ENGINE_VAR_LAGS = 1
ENGINE_FLAT_COST_BPS = 2.0         # simulated panels carry no OHLC -> flat-bps costs
ENGINE_SEED = 21

# --- Multiple testing across the specification grid + Deflated Sharpe ---
RUN_MULTIPLE_TESTING = True
MT_BLOCK_SIZE = 10
MT_N_BOOT = 1000
MT_SEED = 11
SPEC_GRID_NET_ONLY = True          # exclude GROSS specs from the walk-forward / SPA grid (fair vs net EW)

# --- Long-short academic factor comparison (Fama-French via pandas_datareader) ---
RUN_LONGSHORT = True
LONGSHORT_CSV_PATH = None         # optional fallback: CSV of daily factor returns (decimal)
LONGSHORT_COST_BPS = 0.0          # factor returns are notional; set > 0 to stress turnover
LONGSHORT_MATCH_ETF_WINDOW = True  # restrict FF factors to the ETF common-sample window (apples-to-apples)

# --- True out-of-sample walk-forward over the specification grid ---
RUN_WALK_FORWARD = True
WF_MIN_TRAIN_DAYS = 756           # ~3y initial in-sample window
WF_RESELECT_EVERY_DAYS = 63       # re-pick the best spec ~quarterly

# --- Rank / quantile weighting (+ extended-universe breadth test) ---
RUN_QUANTILE_WEIGHTING = True
QUANTILE_TOP_FRAC = 0.4

# Extended single-factor ETF universe for the BREADTH test. Pre-committed rule:
# "one liquid single-factor US large-cap equity ETF per distinct systematic premium,
#  longest-available fund per premium with inception <= 2013."  VERIFY tickers/inceptions
# and liquidity before using; several premia are correlated (Value vs Growth), so the
# number of DISTINCT premia matters more than the count. Adding ETFs shortens the common
# sample to the latest inception (breadth <-> history trade-off).
USE_EXTENDED_UNIVERSE = False      # <<< flip to True to run the whole notebook on the extended set
EXTENDED_TICKERS = {
    "Value": "VLUE", "Size": "SIZE", "Momentum": "MTUM", "Quality": "QUAL",
    "LowVol": "USMV", "DivYield": "VYM", "HighBeta": "SPHB", "Growth": "IWF",
    "Buyback": "PKW", "EqualWeight": "RSP",
}

# --- Conditional (regime) spanning, rolling alpha, static replication, selection mechanism ---
RUN_CONDITIONAL_SPANNING = True
COND_VOL_LOOKBACK = 21
RUN_SELECTION_MECHANISM = True

# --- Universe robustness (leave-one-out + breadth gradient); most informative when extended ---
RUN_UNIVERSE_ROBUSTNESS = True
UNIV_N_RANDOM_SUBSETS = 15
UNIV_SUBSET_SEED = 0

# --- apply the extended universe if requested (re-points the whole pipeline) ---
if USE_EXTENDED_UNIVERSE:
    TICKERS = dict(EXTENDED_TICKERS)
    FACTOR_NAMES = list(TICKERS)
    print(f"Extended universe ON: {len(TICKERS)} factors -> {', '.join(TICKERS)}")

## Load data, build risk-free + benchmark, estimate costs, run the backtest

1. load the raw factor-ETF panel and restrict to the **common-data sample**;
2. build the **historical daily risk-free** series (FRED `DGS3MO`, falling back to Yahoo `^IRX` then to zero if both are unreachable);
3. pull the **SPY** benchmark;
4. estimate daily cost panels under **AR**, **CS** and a **flat-bps** alternative;
5. run the main backtest under the primary estimator and `IMPLEMENTATION_LAG`;
6. report **gross**/**net** metrics (incl. benchmark), turnover (two-way and one-way), and weights.

In [ ]:

# ============================================================
# LOAD DATA
# ============================================================

bundle = None
if DATA_SOURCE == "hdf":
    if HDF_PATH is None:
        raise ValueError("Set HDF_PATH before using DATA_SOURCE='hdf'.")
    r_raw = load_factor_etf_returns_hdf(HDF_PATH, key=HDF_KEY)
    if RUN_TRANSACTION_COSTS:
        bundle = download_factor_etf_bundle(start=str(r_raw.index.min().date()))
elif DATA_SOURCE == "yahoo":
    bundle = download_factor_etf_bundle(start=START_DATE)
    r_raw = bundle["rets"]
else:
    raise ValueError("DATA_SOURCE must be either 'hdf' or 'yahoo'.")

r_raw = r_raw.copy()
r_common = r_raw.dropna(how="any").copy()

print(f"Loaded raw returns panel {r_raw.shape} from {r_raw.index.min().date()} to {r_raw.index.max().date()}.")
display(r_raw.head())
print(f"Common-data backtest sample: {r_common.shape} from {r_common.index.min().date()} to {r_common.index.max().date()}.")

# ============================================================
# MARKET BENCHMARK (passive)
# ============================================================
# EW of the five factor ETFs is itself an active factor bet, not a neutral
# "do nothing". SPY is added as a passive market benchmark for context.
bench_ret = None
if RUN_BENCHMARK:
    bench_ret = download_benchmark_returns(BENCH_TICKER, start=str(r_common.index.min().date()))
    bench_ret = bench_ret.reindex(r_common.index).dropna()
    print(f"Benchmark: {BENCH_TICKER} (buy-and-hold), {len(bench_ret)} aligned observations.")

# ============================================================
# HISTORICAL RISK-FREE SERIES
# ============================================================
if USE_HISTORICAL_RF:
    rf_daily, rf_source_used = build_rf_with_fallback(
        r_common.index, rf_source=RF_SOURCE, day_count=RF_DAY_COUNT, lag_days=RF_LAG_DAYS)
    print(f"Risk-free source in use: {rf_source_used}")
    display(rf_daily.to_frame("rf_daily").head())
else:
    rf_daily = pd.Series(0.0, index=r_common.index, name="rf_daily")
    rf_source_used = "ZERO"
    print("Historical risk-free disabled: using rf_daily = 0.0.")

# ============================================================
# ESTIMATE DAILY TRANSACTION-COST PANELS (AR / CS / FIXED)
# ============================================================
cost_oneway_daily_main = None
spread_full_panels = {}
cost_oneway_panels = {}
tc_mean_table = None
tc_corr_overall = np.nan
tc_corr_by_etf = None

estimators_requested = [TC_ESTIMATOR]
if RUN_TC_ESTIMATOR_COMPARISON:
    estimators_requested = list(dict.fromkeys([TC_ESTIMATOR] + list(TC_ESTIMATORS_TO_COMPARE)))

if RUN_TRANSACTION_COSTS:
    if bundle is None:
        bundle = download_factor_etf_bundle(start=str(r_raw.index.min().date()))
    px_close = bundle["close"].reindex(r_common.index)
    px_high = bundle["high"].reindex(r_common.index)
    px_low = bundle["low"].reindex(r_common.index)

    for est in estimators_requested:
        est = est.upper()
        if est == "AR":
            spread_full = abdi_ranaldo_panel(px_high, px_low, px_close,
                                              tickers=FACTOR_NAMES, clip_upper=TC_SPREAD_CLIP_UPPER)
            spread_full_panels[est] = spread_full.reindex(r_common.index)
            cost_oneway_panels[est] = spread_panel_to_one_way_cost(spread_full).reindex(r_common.index)
        elif est == "CS":
            spread_full = corwin_schultz_panel(px_high, px_low,
                                               tickers=FACTOR_NAMES, clip_upper=TC_SPREAD_CLIP_UPPER)
            spread_full_panels[est] = spread_full.reindex(r_common.index)
            cost_oneway_panels[est] = spread_panel_to_one_way_cost(spread_full).reindex(r_common.index)
        elif est == "FIXED":
            cost_oneway_panels[est] = constant_one_way_cost_panel(r_common.index, FACTOR_NAMES, TC_FIXED_BPS)
        else:
            raise ValueError("Each transaction-cost estimator must be 'AR', 'CS' or 'FIXED'.")

    cost_oneway_daily_main = cost_oneway_panels[TC_ESTIMATOR.upper()]
    print(f"Primary transaction-cost estimator: {TC_ESTIMATOR.upper()}")

    tc_mean_table = pd.concat(
        {est: cost_oneway_panels[est].mean().rename("one_way_cost") for est in cost_oneway_panels}, axis=1)
    print("Mean estimated one-way cost by ETF (bps):")
    display((tc_mean_table * 1e4).round(2))

    if {"AR", "CS"}.issubset(set(spread_full_panels.keys())):
        tc_corr_overall = float(spread_full_panels["AR"].stack().corr(spread_full_panels["CS"].stack()))
        tc_corr_by_etf = pd.Series(
            {fac: float(spread_full_panels["AR"][fac].corr(spread_full_panels["CS"][fac]))
             for fac in FACTOR_NAMES}, name="AR_vs_CS_corr").to_frame()
        print("AR vs CS full-spread correlation (overall):", round(tc_corr_overall, 4))
        display(tc_corr_by_etf.round(4))

# ============================================================
# RUN PRIMARY STRATEGY
# ============================================================
out = run_factor_mom_from_returns(
    r_raw, variant=VARIANT, cost_oneway_daily=cost_oneway_daily_main,
    cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
    charge_initial_trade=TC_CHARGE_INITIAL_TRADE, implementation_lag=IMPLEMENTATION_LAG,
)
rets_gross = out["rets_gross"]
rets_net = out["rets_net"]
signal = out["signal"]
avg_daily_w = out["avg_daily_weights"]
avg_entry_w = out["avg_entry_weights"]
turnover_summary = out["turnover_summary"]
sample_info = out["sample_info"]

# attach the passive benchmark as an extra column for side-by-side reporting
rets_gross_bench = rets_gross.copy()
rets_net_bench = rets_net.copy()
if bench_ret is not None:
    b = bench_ret.reindex(rets_gross.index).rename(BENCH_TICKER)
    rets_gross_bench = pd.concat([rets_gross, b], axis=1)
    rets_net_bench = pd.concat([rets_net, b], axis=1)   # buy-and-hold: gross == net

actual_metrics_gross = perf_table(rets_gross_bench, rf_daily=rf_daily)
actual_metrics_net = perf_table(rets_net_bench, rf_daily=rf_daily)

print("Backtest sample information:")
display(pd.Series(sample_info, name="value").to_frame())
print(f"Implementation lag: {IMPLEMENTATION_LAG} trading day(s).")

print("Gross performance metrics (incl. benchmark):")
display(actual_metrics_gross.round(4))
print(f"Net performance metrics — primary estimator ({TC_ESTIMATOR.upper()}), incl. benchmark:")
display(actual_metrics_net.round(4))
print("Rebalancing / turnover summary (two-way and one-way):")
display(turnover_summary.round(4))
print("Average daily weights:")
display(avg_daily_w.round(4))

# ============================================================
# IN-SAMPLE NET COMPARISON: AR vs CS vs FIXED
# ============================================================
comparison_metrics_net = {}
comparison_turnover = {}
if RUN_TRANSACTION_COSTS and RUN_TC_ESTIMATOR_COMPARISON:
    for est in estimators_requested:
        est_out = run_factor_mom_from_returns(
            r_raw, variant=VARIANT, cost_oneway_daily=cost_oneway_panels[est],
            cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
            charge_initial_trade=TC_CHARGE_INITIAL_TRADE, implementation_lag=IMPLEMENTATION_LAG)
        comparison_metrics_net[est] = perf_table(est_out["rets_net"], rf_daily=rf_daily)
        comparison_turnover[est] = est_out["turnover_summary"]
    print("Net in-sample metrics — estimator comparison (AR vs CS vs FIXED):")
    display(pd.concat(comparison_metrics_net, names=["Estimator", "Strategy"]).round(4))

## Cumulative wealth (gross vs net, with benchmark)

In [ ]:
wealth_gross = wealth_index(rets_gross_bench)
wealth_net = wealth_index(rets_net_bench)

ax = wealth_gross.plot(figsize=(11, 5), title=f"Factor Momentum ({VARIANT}) vs benchmark — gross")
ax.set_ylabel("Wealth index"); plt.show()

ax = wealth_net.plot(figsize=(11, 5),
                     title=f"Factor Momentum ({VARIANT}) vs benchmark — net ({TC_ESTIMATOR.upper()})")
ax.set_ylabel("Wealth index"); plt.show()

## Optional data-driven block-length selection

Average block length chosen by matching the autocorrelation of **absolute returns** of `MomTop1_80_20`. When costs are on, selection uses the **gross** series while the bootstrap resamples the aligned cost panel jointly with returns.

In [ ]:
if RUN_ACF_BLOCK_SELECTION:
    bl = choose_block_length_by_acf_matching(
        rets_gross["MomTop1_80_20"], candidates=ACF_CANDIDATES, nlags=ACF_NLAGS,
        n_boot=ACF_N_BOOT, use_abs=True, seed=BOOTSTRAP_SEED, distance="weighted")
    best_L = bl["best_L"]
    print("Suggested avg_block_len:", best_L)
    display(bl["scores"])
else:
    best_L = AVG_BLOCK_LEN
    print("Using fixed avg_block_len:", best_L)

## Politis-Romano stationary bootstrap

Resamples the **common-sample** factor-ETF panel (and, when costs are on, the cost panel and the benchmark jointly, with shared indices). Inside every replica it rebuilds the signal, the weight schedule, the gross/net backtests and the metrics.

The headline robustness statistic is the **paired difference** (e.g. `MomTop1_80_20 - EW` and `MomTop1_80_20 - SPY`) computed within each replica, together with `P(strat > bench)`.

In [ ]:
PAIRED_METRICS = ("CAGR", "ShR", "SoR", "Calmar", "Martin", "MxDD")

if RUN_BOOTSTRAP:
    boot_primary = bootstrap_factor_mom_metrics(
        r_raw, variant=VARIANT, cost_oneway_daily=cost_oneway_daily_main,
        cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
        charge_initial_trade=TC_CHARGE_INITIAL_TRADE, implementation_lag=IMPLEMENTATION_LAG,
        bench_returns=bench_ret, bench_name=BENCH_TICKER,
        n_boot=N_BOOT, avg_block_len=best_L, seed=BOOTSTRAP_SEED, rf_daily=rf_daily)

    print("Actual backtest metrics — gross")
    display(boot_primary["actual_metrics_gross"].round(4))
    print(f"Actual backtest metrics — net ({TC_ESTIMATOR.upper()})")
    display(boot_primary["actual_metrics_net"].round(4))

    for label, key in [("net CAGR", "CAGR"), ("net Sharpe", "ShR"),
                       ("net Martin", "Martin"), ("net Max drawdown", "MxDD")]:
        print(f"Bootstrap distribution — {label} ({TC_ESTIMATOR.upper()})")
        display(boot_primary["bootstrap_tables_net"][key].round(4))

    # ---- paired difference distributions: this is the real robustness test ----
    mats_net = boot_primary["simulated_metric_vectors_net"]
    print("Paired bootstrap difference — MomTop1_80_20 minus EW (net)")
    display(paired_difference_summary(mats_net, PAIRED_METRICS, "MomTop1_80_20", "EW").round(4))
    print("Paired bootstrap difference — MomTop2_35_35 minus EW (net)")
    display(paired_difference_summary(mats_net, PAIRED_METRICS, "MomTop2_35_35", "EW").round(4))
    if bench_ret is not None:
        print(f"Paired bootstrap difference — MomTop1_80_20 minus {BENCH_TICKER} (net)")
        display(paired_difference_summary(mats_net, PAIRED_METRICS, "MomTop1_80_20", BENCH_TICKER).round(4))
    print("Note: P(strat>bench) is the share of replicas where the strategy beats the "
          "benchmark on each metric (higher-is-better convention; for MxDD 'higher' = shallower).")

    # ---- AR vs CS vs FIXED on net metrics ----
    if RUN_TRANSACTION_COSTS and RUN_TC_ESTIMATOR_COMPARISON:
        boot_comp = {}
        for est in estimators_requested:
            boot_comp[est] = bootstrap_factor_mom_metrics(
                r_raw, variant=VARIANT, cost_oneway_daily=cost_oneway_panels[est],
                cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
                charge_initial_trade=TC_CHARGE_INITIAL_TRADE, implementation_lag=IMPLEMENTATION_LAG,
                bench_returns=bench_ret, bench_name=BENCH_TICKER,
                n_boot=N_BOOT, avg_block_len=best_L, seed=BOOTSTRAP_SEED, rf_daily=rf_daily)
        for label, key in [("CAGR", "CAGR"), ("Sharpe", "ShR"), ("Martin", "Martin")]:
            print(f"Net bootstrap distribution — {label} (AR vs CS vs FIXED)")
            display(pd.concat(
                {est: boot_comp[est]["bootstrap_tables_net"][key] for est in boot_comp},
                names=["Estimator", "Strategy"]).round(4))

## Random-selection (signal-permutation) null

Equal-weighting the five factor ETFs is itself an active factor bet, so beating EW conflates the momentum *signal* with differences in static factor exposure. This section builds a benchmark of strategies that are **structurally identical** to the real one (same 80/20 or 35/35 concentration, same rebalance dates, same costs, same timing) but pick the held factor(s) **at random** each month. The one-sided p-value is the share of random strategies that match or beat the real strategy on each metric: a small p-value is evidence that the **signal**, not the structure, drives performance.

In [ ]:
if RUN_RANDOM_NULL:
    # Structurally-identical, signal-free benchmark: random factor selection with the
    # SAME concentration, rebalance dates, costs and timing as the real strategy.
    # The one-sided p-value is the share of random strategies that match or beat the
    # real one -> isolates the contribution of the momentum SIGNAL from static exposure.
    NULL_METRICS = ("CAGR", "ShR", "SoR", "Calmar", "Martin", "MxDD")

    null_top1 = random_selection_null(
        r_common, signal, w_random_top1_80_20,
        cost_oneway_daily=cost_oneway_daily_main, cost_lookback_days=TC_LOOKBACK_DAYS,
        cost_agg=TC_AGG, charge_initial_trade=TC_CHARGE_INITIAL_TRADE,
        implementation_lag=IMPLEMENTATION_LAG, metrics=NULL_METRICS,
        use_net=RUN_TRANSACTION_COSTS, n_random=N_RANDOM, seed=RANDOM_NULL_SEED, rf_daily=rf_daily)

    null_top2 = random_selection_null(
        r_common, signal, w_random_top2_35_35,
        cost_oneway_daily=cost_oneway_daily_main, cost_lookback_days=TC_LOOKBACK_DAYS,
        cost_agg=TC_AGG, charge_initial_trade=TC_CHARGE_INITIAL_TRADE,
        implementation_lag=IMPLEMENTATION_LAG, metrics=NULL_METRICS,
        use_net=RUN_TRANSACTION_COSTS, n_random=N_RANDOM, seed=RANDOM_NULL_SEED, rf_daily=rf_daily)

    actual_net = perf_table(rets_net, rf_daily=rf_daily)
    print("Random-selection null — MomTop1_80_20 vs random top-1 (same 80/20 structure)")
    display(null_pvalues(actual_net.loc["MomTop1_80_20"], null_top1, NULL_METRICS).round(4))
    print("Random-selection null — MomTop2_35_35 vs random top-2 (same 35/35 structure)")
    display(null_pvalues(actual_net.loc["MomTop2_35_35"], null_top2, NULL_METRICS).round(4))

## Alternative weighting: separating selection from concentration

The fixed 80/20 and 35/35 schemes bundle two decisions — *which* factors to hold and *how concentrated* to be. Re-running the same momentum **selection** with **inverse-volatility** sizing isolates the selection effect from the concentration effect; `InvVol_all` is also a more neutral, risk-parity-like blend than equal weight (which ignores the factors' very different volatilities).

In [ ]:
if RUN_ALT_WEIGHTING:
    # Inverse-volatility sizing separates SELECTION from CONCENTRATION: same momentum
    # picks, but risk-based weights instead of the fixed 80/20 or 35/35. w_inverse_vol_all
    # is also a more neutral (risk-parity-like) blend than equal weight.
    w_iv_top1 = w_topk_inverse_vol(signal, r_common, k=1, lookback=ALT_WEIGHT_LOOKBACK)
    w_iv_top2 = w_topk_inverse_vol(signal, r_common, k=2, lookback=ALT_WEIGHT_LOOKBACK)
    w_iv_all = w_inverse_vol_all(signal, r_common, lookback=ALT_WEIGHT_LOOKBACK)
    alt = {}
    for nm, wm in [("MomTop1_invvol", w_iv_top1), ("MomTop2_invvol", w_iv_top2),
                   ("InvVol_all", w_iv_all)]:
        _, n = run_named_strategy(
            r_common, wm, cost_oneway_daily=cost_oneway_daily_main,
            cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
            charge_initial_trade=TC_CHARGE_INITIAL_TRADE, implementation_lag=IMPLEMENTATION_LAG)
        alt[nm] = n
    compare = pd.concat([rets_net[["MomTop1_80_20", "MomTop2_35_35", "EW"]],
                         pd.concat(alt, axis=1)], axis=1)
    print("Net metrics — fixed concentration (80/20, 35/35, EW) vs inverse-volatility sizing:")
    display(perf_table(compare, rf_daily=rf_daily).round(4))

## Serial-dependence diagnostics (does the bootstrap engine matter?)

Ljung-Box on **levels** captures the weak autoregressive / short-term-reversal structure in the mean; Ljung-Box on **squares** and the **ARCH-LM** test capture volatility clustering. Strong clustering is the case where the stationary bootstrap (which shuffles short blocks) understates long, clustered drawdowns — and where the **FHS engine** below is the more honest choice.

In [ ]:
if RUN_DEPENDENCE_DIAGNOSTICS:
    # If squared returns are strongly autocorrelated (ARCH), the stationary bootstrap
    # with short blocks understates clustered crashes -> prefer the FHS engine below.
    # Levels probe the (weak) AR / short-term-reversal structure in the mean.
    diag_series = {
        "MomTop1_net": rets_net["MomTop1_80_20"],
        "EW_net": rets_net["EW"],
        "Mom_minus_EW": (rets_net["MomTop1_80_20"] - rets_net["EW"]).dropna(),
    }
    if bench_ret is not None:
        diag_series[BENCH_TICKER] = bench_ret.reindex(rets_net.index).dropna()
    diag = dependence_diagnostics(diag_series, lags=DIAG_LAGS)
    print(f"Serial-dependence diagnostics (lags={DIAG_LAGS}) — Ljung-Box (levels & squares) + ARCH-LM:")
    display(diag.round(4))
    print("Reading: small LB_levels_p -> autocorrelation in the mean (AR / reversal); "
          "small LB_squares_p / ARCH_LM_p -> volatility clustering, which motivates the FHS engine.")

## Spanning regression (is the strategy just static factor exposure?)

Regressing the strategy's excess return on the five factor ETFs' excess returns (plus `SPY`) with **Newey-West** standard errors gives the cleanest benchmark of all: a **zero alpha** means the strategy is *spanned* by a fixed combination of the factors, so the timing/selection adds nothing beyond static exposure. This also sidesteps the `SPY` vs `SPY − rf` question, since everything is in excess returns.

In [ ]:
if RUN_SPANNING:
    # Regress the strategy's EXCESS return on the five factor ETFs' excess returns (+ SPY),
    # Newey-West HAC errors. An alpha indistinguishable from zero means the strategy is
    # "spanned" by a static factor combination: the timing/selection adds nothing.
    for strat in ["MomTop1_80_20", "MomTop2_35_35"]:
        sp = spanning_regression(rets_net[strat], r_common, rf_daily=rf_daily,
                                 extra_returns=bench_ret)
        print(f"Spanning regression — {strat} (net): Newey-West lags={sp['nw_lags']}, n={sp['n']}")
        print(f"  alpha (annualised) = {sp['alpha_annual']:.4f}   t = {sp['alpha_t']:.2f}   "
              f"p = {sp['alpha_p']:.3f}   R^2 = {sp['r2']:.3f}")
        display(sp["coefficients"].round(4))

## Sharpe-ratio inference: Ledoit-Wolf delta-Sharpe and PSR

The **Ledoit-Wolf (2008)** test compares two Sharpe ratios under non-iid returns (HAC standard error plus a studentized circular-block bootstrap), which is the correct way to ask whether `MomTop1_80_20` beats `EW`/`SPY` on a *risk-adjusted* basis. The **Probabilistic Sharpe Ratio** reports the probability the true Sharpe exceeds zero, correcting for skewness and fat tails. The search-corrected **Deflated Sharpe** appears in the multiple-testing section.

In [ ]:
if RUN_SHARPE_INFERENCE:
    # Ledoit-Wolf (2008): HAC + studentized circular-block bootstrap test for the
    # DIFFERENCE of two Sharpe ratios under non-iid returns (both series in excess units).
    def _exc(s):
        s = pd.Series(s).dropna()
        return s - rf_daily.reindex(s.index)
    benches = {"EW": rets_net["EW"]}
    if bench_ret is not None:
        benches[BENCH_TICKER] = bench_ret
    for bname, bser in benches.items():
        lw = sharpe_diff_test(_exc(rets_net["MomTop1_80_20"]), _exc(bser),
                              n_boot=LW_N_BOOT, seed=BOOTSTRAP_SEED)
        print(f"Delta-Sharpe (Ledoit-Wolf) — MomTop1_80_20 minus {bname}: "
              f"dSR_ann = {lw['delta_ann']:.3f}   t = {lw['tstat']:.2f}   p = {lw['pval']:.3f}   "
              f"(HAC bw = {lw['hac_bw']}, block = {lw['block_size']}, boot = {lw['n_boot']})")
    psr = probabilistic_sharpe_ratio(rets_net["MomTop1_80_20"], sr_benchmark=0.0)
    print(f"Probabilistic Sharpe (MomTop1_80_20 vs SR=0): PSR = {psr['psr']:.4f}  "
          f"(skew = {psr['skew']:.2f}, kurtosis = {psr['kurtosis']:.2f}, SR_ann = {psr['sharpe_ann']:.3f})")
    print("Deflated Sharpe (corrected for the specification search) is reported in the multiple-testing cell.")

## Break-even transaction cost

A single, robust summary of cost sensitivity: the flat one-way cost (in bps) at which the edge over `EW`/`SPY` disappears. Because it is a level rather than a path, it sidesteps the noisy AR/CS spread estimates — compare it against the per-trade costs estimated earlier.

In [ ]:
if RUN_BREAKEVEN:
    # The flat one-way cost (bps) that drives the edge to zero -- a single, interpretable
    # number that sidesteps the noisy AR/CS spread estimates.
    be_rows = {}
    for strat in ["MomTop1_80_20", "MomTop2_35_35"]:
        for mode, label in [("cagr_vs_ew", "CAGR vs EW"), ("sharpe_vs_ew", "Sharpe vs EW")]:
            be = breakeven_cost_bps(
                r_raw, variant=VARIANT, strat=strat, mode=mode, rf_daily=rf_daily,
                c_max_bps=BREAKEVEN_C_MAX_BPS, implementation_lag=IMPLEMENTATION_LAG,
                cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
                charge_initial_trade=TC_CHARGE_INITIAL_TRADE)
            be_rows[f"{strat} | {label}"] = be.get("breakeven_bps", float("nan"))
        if bench_ret is not None:
            be = breakeven_cost_bps(
                r_raw, variant=VARIANT, strat=strat, mode="cagr_vs_spy", bench_returns=bench_ret,
                rf_daily=rf_daily, c_max_bps=BREAKEVEN_C_MAX_BPS, implementation_lag=IMPLEMENTATION_LAG,
                cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
                charge_initial_trade=TC_CHARGE_INITIAL_TRADE)
            be_rows[f"{strat} | CAGR vs {BENCH_TICKER}"] = be.get("breakeven_bps", float("nan"))
    print(f"Break-even one-way cost in bps (edge vanishes at/above this cost; cap = {BREAKEVEN_C_MAX_BPS}):")
    display(pd.Series(be_rows, name="breakeven_oneway_bps").to_frame().round(2))
    print("Compare against the estimated per-trade costs printed earlier (AR/CS/FIXED).")

## BCa intervals on the realized statistics

Bias-corrected and accelerated intervals (circular-block bootstrap + block jackknife) for the realized Sharpe and CAGR. For asymmetric statistics these are better calibrated than the raw percentile intervals reported in the stationary-bootstrap section.

In [ ]:
if RUN_BCA:
    # Bias-corrected & accelerated intervals on the realized P&L (circular-block bootstrap
    # + block jackknife): better calibrated than raw percentiles for skewed statistics
    # such as Sharpe and Martin.
    bca_block = max(2, int(round(optimal_block_length_ppw(rets_net["MomTop1_80_20"])["stationary"])))
    print(f"BCa 95% intervals for MomTop1_80_20 (net): block size = {bca_block}, n_boot = {BCA_N_BOOT}")
    bca_rows = {}
    for mname, mfn in [("Sharpe", _metric_sharpe), ("CAGR", _metric_cagr)]:
        ci = bca_interval(rets_net["MomTop1_80_20"], mfn, block_size=bca_block,
                          n_boot=BCA_N_BOOT, alpha=0.05, seed=BOOTSTRAP_SEED)
        bca_rows[mname] = {"estimate": ci["theta"], "lo95": ci["lo"], "hi95": ci["hi"]}
    display(pd.DataFrame(bca_rows).T.round(4))

## Regime / subsample stability

Momentum is regime-dependent, so a single full-sample number can hide where the edge actually lives. Performance is broken out by calendar year, by sample halves, and by the trailing **market-volatility regime** (using `SPY`'s realized vol as the regime variable; swap in VIX if available).

In [ ]:
if RUN_REGIME:
    target = "MomTop1_80_20"
    print(f"{target} (net) — performance by calendar year:")
    display(metrics_by_calendar_year(rets_net[target], rf_daily=rf_daily).round(4))
    print(f"{target} (net) — performance by sample halves:")
    display(metrics_two_halves(rets_net[target], rf_daily=rf_daily).round(4))
    if bench_ret is not None:
        print(f"{target} (net) — by market volatility regime (trailing {REGIME_VOL_LOOKBACK}d vol of {BENCH_TICKER}):")
        display(metrics_by_vol_regime(rets_net[target], bench_ret, lookback=REGIME_VOL_LOOKBACK,
                                      n_regimes=REGIME_N, rf_daily=rf_daily).round(4))

## Dependence-aware bootstrap engines (FHS and VAR-sieve)

These engines address the concern that the stationary bootstrap, by reshuffling short blocks, breaks the volatility clustering and clustered crashes that drive momentum's left tail. **FHS** filters each factor with an AR(p)-GJR-GARCH(1,1), resamples the standardized-residual **rows** (preserving contemporaneous cross-factor correlation) and re-inflates through each series' own volatility recursion. The **VAR-sieve wild bootstrap** preserves cross-factor lead-lag and is robust to heteroskedasticity. Output mirrors the stationary bootstrap, so the same paired-difference reading applies. *FHS requires the `arch` package.*

In [ ]:
if RUN_ENGINE_BOOTSTRAP:
    # Dependence-aware engines that GENERATE panels (rather than resampling blocks):
    #   * FHS = AR(p)-GJR-GARCH filtering + row-resampled standardized residuals,
    #           preserving volatility clustering and fat (asymmetric) left tails;
    #   * VAR-sieve = VAR(p) + recursive WILD bootstrap, preserving cross-factor lead-lag.
    # The whole strategy is rebuilt inside every replica (costs via a flat-bps panel,
    # since simulated panels carry no OHLC). Requires the `arch` package for FHS.
    engine_boot = {}
    for eng in ENGINES:
        engine_boot[eng] = bootstrap_metrics_via_engine(
            r_common, engine=eng, variant=VARIANT, bench_returns=bench_ret, bench_name=BENCH_TICKER,
            flat_cost_oneway_bps=(ENGINE_FLAT_COST_BPS if RUN_TRANSACTION_COSTS else None),
            ar_lags=ENGINE_AR_LAGS, var_lags=ENGINE_VAR_LAGS,
            n_sims=ENGINE_N_SIMS, seed=ENGINE_SEED, rf_daily=rf_daily,
            implementation_lag=IMPLEMENTATION_LAG)
    for eng in ENGINES:
        mats = engine_boot[eng]["simulated_metric_vectors_net"]
        print(f"[{eng.upper()}] net Sharpe distribution")
        display(engine_boot[eng]["bootstrap_tables_net"]["ShR"].round(4))
        print(f"[{eng.upper()}] net Martin distribution")
        display(engine_boot[eng]["bootstrap_tables_net"]["Martin"].round(4))
        print(f"[{eng.upper()}] paired difference MomTop1_80_20 - EW (net)")
        display(paired_difference_summary(mats, PAIRED_METRICS, "MomTop1_80_20", "EW").round(4))
        if bench_ret is not None:
            print(f"[{eng.upper()}] paired difference MomTop1_80_20 - {BENCH_TICKER} (net)")
            display(paired_difference_summary(mats, PAIRED_METRICS, "MomTop1_80_20", BENCH_TICKER).round(4))
    print("Compare these intervals with the stationary-bootstrap ones: if the edge shrinks "
          "under FHS, it depended on the volatility clustering the stationary bootstrap broke up.")

## Multiple testing under specification search

Reporting the best of several specifications inflates significance. **White's (2000) Reality Check** and **Hansen's (2005) SPA** test the null that the best specification does **not** beat the benchmark once the whole search (variant × concentration × cost) is accounted for; the **Deflated Sharpe Ratio (Bailey & Lopez de Prado 2014)** corrects the best spec's Sharpe for the number of trials and for non-normality. These directly answer the data-snooping objection.

In [ ]:
if RUN_MULTIPLE_TESTING:
    # Across the grid (variant x concentration x cost), is the BEST specification really
    # better than the benchmark once the search is accounted for?  White (2000) Reality
    # Check + Hansen (2005) SPA, plus the Deflated Sharpe Ratio on the best spec.
    mt_cost_panels = {}
    if RUN_TRANSACTION_COSTS:
        for est in estimators_requested:
            mt_cost_panels[est] = cost_oneway_panels[est]
    if (not SPEC_GRID_NET_ONLY) or (not mt_cost_panels):
        mt_cost_panels = {"GROSS": None, **mt_cost_panels}
    spec_mat, sr_trials = build_spec_return_matrix(
        r_raw, variants=("6-1", "12-1"), concentrations=("top1", "top2"),
        cost_panels=mt_cost_panels, rf_daily=rf_daily, implementation_lag=IMPLEMENTATION_LAG,
        cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
        charge_initial_trade=TC_CHARGE_INITIAL_TRADE)
    bench_for_mt = rets_net["EW"]
    rc = reality_check_white(spec_mat, bench_for_mt, block_size=MT_BLOCK_SIZE, n_boot=MT_N_BOOT, seed=MT_SEED)
    spa = spa_test_hansen(spec_mat, bench_for_mt, block_size=MT_BLOCK_SIZE, reps=MT_N_BOOT, seed=MT_SEED)
    print(f"Specification grid: K = {spec_mat.shape[1]} strategies vs EW benchmark, n = {rc['n']}")
    print(f"White Reality Check   p = {rc['pvalue']:.4f}   (best spec: {rc['best_spec']})")
    print(f"Hansen SPA p-values:  " + ", ".join(f"{k}={v:.4f}" for k, v in spa['pvalues'].items())
          + "   (quote the 'consistent' one)")
    print("Mean annualised excess-over-EW by specification:")
    display(rc["mean_excess_ann"].round(4).to_frame("mean_excess_ann"))
    dsr = deflated_sharpe_ratio(spec_mat[rc["best_spec"]], list(sr_trials.values()))
    print(f"Deflated Sharpe Ratio (best spec '{rc['best_spec']}'): DSR = {dsr['dsr']:.4f}   "
          f"SR_ann = {dsr['sharpe_ann']:.3f} vs threshold SR0_ann = {dsr['sr0_threshold_ann']:.3f}   "
          f"(N_trials = {dsr['n_trials']})")
    _sr_std = float(np.std([s for s in sr_trials.values() if np.isfinite(s)], ddof=1))
    print(f"  CAVEAT: the {spec_mat.shape[1]} specs are highly correlated (per-obs trial-Sharpe "
          f"std = {_sr_std:.4f}); the DSR's independent-trials assumption is then violated and it "
          f"UNDER-deflates (SR0 collapses toward 0). Treat the SPA 'consistent' p-value above as "
          f"the multiplicity-robust verdict, not the DSR.")

## Long-short academic factors: localising the gap

The decisive economic test. Factor momentum is documented on **long-short** factor returns (Gupta-Kelly; Ehsani-Linnainmaa), not on long-only ETF wrappers. This section runs the *same* signal on the Fama-French long-short factors (SMB, HML, RMW, CMA, Mom). If momentum beats the equal-weight factor blend on the **long-short** factors while it is spanned in the **ETF** version, the implementation gap is the *vehicle* (market beta plus a thin long-only tilt), not the signal. Requires network access to Kenneth French's data library (or a fallback CSV via `LONGSHORT_CSV_PATH`).

In [ ]:
if RUN_LONGSHORT:
    # Localise the gap: run the SAME cross-sectional momentum signal on the LONG-SHORT
    # academic factors (Fama-French SMB/HML/RMW/CMA/Mom). If momentum works there but is
    # spanned in the ETF wrappers, the gap is the VEHICLE (market beta + thin long-only tilt),
    # not the signal. Requires network access to French's library (or a fallback CSV).
    ls_rets = ls_rf = ls_mkt = None
    try:
        ls_rets, ls_rf, ls_mkt = download_famafrench_factors(start=START_DATE)
        print(f"Fama-French long-short factors: {ls_rets.shape}, "
              f"{ls_rets.index.min().date()} to {ls_rets.index.max().date()} "
              f"({', '.join(ls_rets.columns)})")
    except Exception as e:
        if LONGSHORT_CSV_PATH:
            ls_rets, ls_rf, ls_mkt = load_factor_returns_csv(LONGSHORT_CSV_PATH, rf_col="RF", mkt_col="Mkt-RF")
            print(f"FF download failed ({type(e).__name__}); loaded factors from {LONGSHORT_CSV_PATH}.")
        else:
            print(f"Fama-French factors unavailable ({type(e).__name__}). Set LONGSHORT_CSV_PATH "
                  f"to a CSV of daily factor returns to run this section.")

    if ls_rets is not None:
        if LONGSHORT_MATCH_ETF_WINDOW:
            lo, hi = r_common.index.min(), r_common.index.max()
            ls_rets = ls_rets.loc[lo:hi]
            if ls_rf is not None: ls_rf = ls_rf.loc[lo:hi]
            if ls_mkt is not None: ls_mkt = ls_mkt.loc[lo:hi]
            print(f"Long-short factors restricted to the ETF window {lo.date()} -> {hi.date()} "
                  f"({len(ls_rets)} days) for an apples-to-apples comparison.")
        ls_rf_use = ls_rf if ls_rf is not None else rf_daily
        ls_common = ls_rets.dropna(how="any")
        ls_out = run_factor_mom_on_panel(ls_rets, variant=VARIANT,
                                         implementation_lag=IMPLEMENTATION_LAG, cost_oneway_bps=LONGSHORT_COST_BPS)
        ls_tab = perf_table(ls_out["rets"], rf_daily=ls_rf_use)
        print("Long-short factor-momentum metrics:")
        display(ls_tab.round(4))

        # paired bootstrap MomTop1 - EW on the long-short panel (rebuilt per replica)
        LSM = ("CAGR", "ShR", "Martin", "MxDD")
        mats = {(m, s): [] for m in LSM for s in ("MomTop1", "EW")}
        T = len(ls_common)
        for ii in stationary_bootstrap_index_stream(T, best_L, N_BOOT, BOOTSTRAP_SEED):
            sim = ls_common.iloc[ii].copy(); sim.index = ls_common.index
            tb = perf_table(run_factor_mom_on_panel(sim, variant=VARIANT,
                            implementation_lag=IMPLEMENTATION_LAG)["rets"], rf_daily=ls_rf_use)
            for m in LSM:
                for s in ("MomTop1", "EW"):
                    mats[(m, s)].append(tb.loc[s, m] if (s in tb.index and m in tb.columns) else np.nan)
        mats = {k: np.asarray(v, float) for k, v in mats.items()}
        print("Long-short paired bootstrap — MomTop1 minus EW:")
        display(paired_difference_summary(mats, LSM, "MomTop1", "EW").round(4))

        # signal-permutation null on the long-short panel (same 80/20 structure, random pick)
        rngp = np.random.default_rng(RANDOM_NULL_SEED)
        null = {m: [] for m in LSM}
        for _ in range(N_RANDOM):
            wr = w_random_top1_80_20(ls_out["signal"].index, list(ls_out["signal"].columns), rngp)
            mm = perf_metrics(backtest_returns_panel_monthly(ls_common, wr,
                              implementation_lag=IMPLEMENTATION_LAG), rf_daily=ls_rf_use)
            for m in LSM:
                null[m].append(mm.get(m, np.nan))
        null = {m: np.asarray(v, float) for m, v in null.items()}
        print("Long-short signal-permutation null — MomTop1 vs random top-1:")
        display(null_pvalues(ls_tab.loc["MomTop1"], null, LSM).round(4))

        # side-by-side: ETF wrappers vs long-short factors (the localisation of the gap)
        etf_tab = perf_table(rets_net, rf_daily=rf_daily)
        cmp = pd.DataFrame({
            "ETF_MomTop1": etf_tab.loc["MomTop1_80_20", ["CAGR", "ShR", "Martin"]],
            "ETF_EW": etf_tab.loc["EW", ["CAGR", "ShR", "Martin"]],
            "LS_MomTop1": ls_tab.loc["MomTop1", ["CAGR", "ShR", "Martin"]],
            "LS_EW": ls_tab.loc["EW", ["CAGR", "ShR", "Martin"]],
        })
        print("ETF wrappers vs long-short factors:")
        display(cmp.round(4))
        print("If LS_MomTop1 beats LS_EW while ETF_MomTop1 does not, the edge lives in the "
              "long-short factor construction, not in the long-only ETF implementation.")

## True out-of-sample walk-forward

A different *kind* of evidence than the in-sample SPA test: an expanding-window protocol that re-picks the best specification on past data only and banks its future returns. If the OOS-selected strategy does not beat equal-weight on the same window, the specification search does not survive out of sample.

In [ ]:
if RUN_WALK_FORWARD:
    # True OOS: expanding window, re-pick the best spec by IN-SAMPLE Sharpe ~quarterly, then
    # bank that spec's next-quarter OOS return. Answers the spec-search-overfit worry directly.
    # The grid is NET-only (SPEC_GRID_NET_ONLY) so the OOS pick is compared fairly to net EW.
    _wf_panels = {}
    if RUN_TRANSACTION_COSTS:
        for est in estimators_requested:
            _wf_panels[est] = cost_oneway_panels[est]
    if (not SPEC_GRID_NET_ONLY) or (not _wf_panels):
        _wf_panels = {"GROSS": None, **_wf_panels}
    spec_mat_wf, _ = build_spec_return_matrix(
        r_raw, variants=("6-1", "12-1"), concentrations=("top1", "top2"),
        cost_panels=_wf_panels, rf_daily=rf_daily, implementation_lag=IMPLEMENTATION_LAG,
        cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG, charge_initial_trade=TC_CHARGE_INITIAL_TRADE)
    print(f"Walk-forward spec grid ({spec_mat_wf.shape[1]} specs): {', '.join(spec_mat_wf.columns)}")
    wf = walk_forward_select(spec_mat_wf, min_train_days=WF_MIN_TRAIN_DAYS,
                             reselect_every_days=WF_RESELECT_EVERY_DAYS, rf_daily=rf_daily)
    print(f"Walk-forward OOS: {wf['n_reselections']} reselections, n = {wf['oos_metrics'].get('n')}")
    ew_oos = perf_metrics(rets_net["EW"].reindex(wf["oos_returns"].index), rf_daily=rf_daily)
    comp = pd.DataFrame({"OOS_selected": pd.Series(wf["oos_metrics"]),
                         "EW_same_window": pd.Series(ew_oos)}).T[
        ["CAGR", "ShR", "SoR", "Calmar", "Martin", "MxDD"]]
    display(comp.round(4))
    print("Specifications chosen over time:")
    display(wf["chosen"]["spec"].value_counts().to_frame("times_selected"))
    print("If the OOS-selected line does not beat EW on the same window, the spec search does "
          "not add value out-of-sample (consistent with the SPA result).")

## Rank / quantile weighting (and extended-universe hook)

Inverse-volatility sizing (earlier) and **rank/quantile** weighting here separate *selection* from the arbitrary 80/20 concentration. The builders scale to any universe size; set `EXTENDED_TICKERS` (8-12 distinct single-factor ETFs chosen by a pre-committed rule) and re-run from the data cell to test whether the negative result is an artefact of `N = 5`.

In [ ]:
if RUN_QUANTILE_WEIGHTING:
    # Rank/quantile sizing separates SELECTION from the fixed 80/20 concentration and is the
    # natural construction for a larger universe.
    qw = {"MomQ_top": w_quantile_long(signal, top_frac=QUANTILE_TOP_FRAC),
          "MomRank": w_rank_proportional(signal),
          "EW": w_equal(signal)}
    qres = {}
    for nm, wm in qw.items():
        _, nnet = run_named_strategy(r_common, wm, cost_oneway_daily=cost_oneway_daily_main,
                                     cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
                                     charge_initial_trade=TC_CHARGE_INITIAL_TRADE, implementation_lag=IMPLEMENTATION_LAG)
        qres[nm] = nnet
    print(f"Rank/quantile weighting on the {signal.shape[1]}-factor universe (top_frac={QUANTILE_TOP_FRAC}, net):")
    display(perf_table(pd.concat(qres, axis=1), rf_daily=rf_daily).round(4))
    if USE_EXTENDED_UNIVERSE:
        print(f"Extended universe is ACTIVE ({signal.shape[1]} factors): every table in the notebook "
              f"is computed on it, and rank/quantile sizing is the natural construction at this N.")
    else:
        print(f"BREADTH TEST: set USE_EXTENDED_UNIVERSE = True in the config cell to re-point the WHOLE "
              f"notebook onto the {len(EXTENDED_TICKERS)}-ETF extended universe "
              f"({', '.join(list(EXTENDED_TICKERS)[:5])}, ...). Verify tickers/inceptions first; the "
              f"common sample shortens to the latest inception.")

## Conditional (regime) spanning and rolling alpha

The full-sample spanning alpha is zero, but momentum is regime-dependent. A two-alpha spanning regression (interacting the constant and factor loadings with a high/low market-volatility dummy, Newey-West) tests whether a regime-conditional alpha is hidden inside the unconditional zero; the rolling 2-year alpha shows whether any alpha is episodic.

In [ ]:
if RUN_CONDITIONAL_SPANNING:
    # Does the zero full-sample alpha hide a regime-conditional alpha? Two-alpha spanning with
    # a high/low market-volatility dummy (Newey-West), plus the rolling alpha for inspection.
    reg = vol_regime_dummy(bench_ret if bench_ret is not None else rets_net["EW"],
                           rets_net.index, lookback=COND_VOL_LOOKBACK, q=0.5)
    for strat in ["MomTop1_80_20", "MomTop2_35_35"]:
        csp = spanning_regression_conditional(rets_net[strat], r_common, reg,
                                              rf_daily=rf_daily, extra=bench_ret)
        print(f"Conditional spanning — {strat} (net): NW lags={csp['nw_lags']}, "
              f"n_low={csp['n_low']}, n_high={csp['n_high']}")
        display(pd.DataFrame({"low_vol": csp["low_regime"], "high_vol": csp["high_regime"],
                              "high_minus_low": csp["delta_high_minus_low"]}).T.round(4))
    ra = rolling_spanning_alpha(rets_net["MomTop1_80_20"], r_common, rf_daily=rf_daily,
                                window=504, extra=bench_ret)
    ax = ra.plot(figsize=(11, 4), title="MomTop1_80_20 — rolling 2y annualised spanning alpha (net)")
    ax.axhline(0, color="k", lw=0.8); ax.set_ylabel("alpha (annualised)"); plt.show()

## Spanning decomposition and the mechanism behind the positive finding

Two complementary views. The **static replication** rebuilds MomTop1 from its fixed spanning betas (alpha excluded) and reports how tightly that constant factor blend tracks the strategy — making "spanned" concrete. The **selection-quality** diagnostic asks *why* momentum beats the random-selection null on drawdowns but not on the mean: it ranks the forward return of the picked factor among all factors and splits by regime. A pick that avoids the worst factor (especially in high-volatility months) is a *defensive* edge — it improves the tail, not the average.

In [ ]:
if RUN_SELECTION_MECHANISM:
    # (1) Static replication: MomTop1 is (approximately) a FIXED factor blend.
    rep = static_replication(rets_net["MomTop1_80_20"], r_common, rf_daily=rf_daily, extra=bench_ret)
    print(f"Static replication of MomTop1_80_20: corr(static, actual) = {rep['corr_static_vs_actual']:.3f}, "
          f"R^2 = {rep['r2']:.3f}, alpha_ann = {rep['alpha_ann']:.4f}; "
          f"static Sharpe {rep['static_sharpe_ann']:.3f} vs actual {rep['actual_sharpe_ann']:.3f}")
    print("Implied long-only factor weights of the static replica:")
    display(rep["implied_long_weights"].round(3).to_frame("weight"))

    # (2) Mechanism for the positive finding: does momentum AVOID the worst factor, more so
    #     in high-volatility months?
    reg = vol_regime_dummy(bench_ret if bench_ret is not None else rets_net["EW"],
                           r_common.index, lookback=COND_VOL_LOOKBACK, q=0.5)
    sq = selection_quality_by_regime(signal, r_common, regime=reg, k=1)
    print(f"Selection quality of the top-1 momentum pick (rank 1 = worst .. {r_common.shape[1]} = best):")
    display(sq["summary"].round(4))
    print("If P_worst sits below the random 1/N and the mean rank above (N+1)/2 — more so in "
          "high-vol — the edge is DEFENSIVE: momentum avoids the worst factor, improving drawdowns, "
          "which matches the permutation null winning on Martin/Calmar but not on the mean.")

## Universe robustness: is the breadth result real or a lucky universe?

The extended-breadth finding depends on the chosen ETF set, some of whose members are debatable as factors. This section stress-tests it two ways: **leave-one-out** (drop each ETF and recompute the net Sharpe gap of MomTop1 over equal-weight) and a **breadth gradient** (the gap across random subsets of increasing size). A gap that survives dropping any single name and rises with the number of factors is evidence the effect is genuinely breadth-driven; a gap that hinges on one ETF is a red flag of a single-name artefact.

In [ ]:
if RUN_UNIVERSE_ROBUSTNESS and r_common.shape[1] > 5:
    # Sensitivity to the CHOICE of universe (researcher degrees of freedom) and the BREADTH
    # gradient. Leave-one-out drops each factor in turn; the size sweep draws random subsets
    # of N factors. A gap that survives dropping any single (possibly dubious) ETF, and that
    # rises with N, is the clean evidence the effect is breadth-driven rather than an artefact.
    ur = universe_robustness(
        r_common, cost_oneway_daily=cost_oneway_daily_main, rf_daily=rf_daily, variant=VARIANT,
        implementation_lag=IMPLEMENTATION_LAG, cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
        charge_initial_trade=TC_CHARGE_INITIAL_TRADE, strat="MomTop1_80_20", bench="EW",
        n_random_subsets=UNIV_N_RANDOM_SUBSETS, subset_seed=UNIV_SUBSET_SEED)
    print("Leave-one-out — net Sharpe of MomTop1 / EW and their gap, dropping each factor "
          "(__FULL__ = all factors):")
    display(ur["leave_one_out"].round(4))
    print("Breadth gradient — random subsets of N factors: mean net Sharpe of MomTop1 and the "
          "MomTop1-minus-EW Sharpe gap (5th/95th pct, and fraction of subsets with a positive gap):")
    display(ur["size_sweep"].round(4))
    print("Read: a gap rising with N (and gap_frac_pos increasing) => breadth-driven effect; a gap "
          "that collapses when one ETF is dropped => fragile, single-name artefact. Robustness here "
          "directly addresses the universe-choice caveat behind the extended-breadth result.")
else:
    print("Universe robustness runs on the extended universe only "
          "(needs > 5 factors; set USE_EXTENDED_UNIVERSE = True and RUN_UNIVERSE_ROBUSTNESS = True).")